# Extension: training and freezing the acoustic-label control arm A (paired data only)

Trains the control network **A** (the T architecture with S1 and S2 heads and no R head) on the paired training pack
with labels derived from the phonocardiogram alone (Springer segmenter, no electrocardiogram), three seeds, with the
crops, augmentation, schedule and selection rule of T, as fixed in `Next Study - Beat-Accurate Synthetic ECG/EXP2_PREREG.md`,
and freezes it (weights hashed, freeze record written). Built by `extension_study/notebooks/build_train_ac.py`; the
study code is embedded and checked by SHA-256. **This notebook never reads CirCor.**

**How to run.** Runtime > Change runtime type > A100 GPU, High-RAM. Leave `SMOKE = False`; Runtime > Run all.
Expected time: about 30 minutes per model, three models.

**Inputs on Drive.** `MyDrive/JBHI-Revision/Extension_Study/corpus/paired_v3/` (the pack of T) and
`MyDrive/JBHI-Revision/Extension_Study/corpus/labels_acoustic_v1/` holding `events_paired_v3.npz` and `laws.json` (upload the two
files from `PhD Level Research/derived/labels_acoustic_v1/`; their hashes are checked).
**Outputs on Drive.** `MyDrive/JBHI-Revision/Extension_Study/checkpoints/<timestamp>_ac/`: one `.pt` per model with its
training history, `freeze.json`, and `paired_test.json` (Gate A, descriptive).

In [ ]:
SMOKE = False                    # True: one short epoch, nothing frozen (pipeline test)
KINDS = ('A',)
SEEDS = {'A': (11, 22, 33)}                 # three seeds, the seeds of T
EXP2_PLAN_SHA256 = 'f0b9fe2b9ddc76a2ffaa0c170cf13d0392e5e68f7a65a4dcdfbf190e5a31a184'   # Next Study - Beat-Accurate Synthetic ECG/EXP2_PREREG.md
AC_VERSION = 'labels_acoustic_v1'
EVENTS_FILE, EVENTS_SHA256 = 'events_paired_v3.npz', '663277f316cce20347a577acb672181b12430b2433a25e7a0a92625b8e3173b4'
LAWS_SHA256 = '850c77aceb747522f4d73e695f927f67f19c98394c8e5afb8a08790cbe603720'
LABEL_GATE = {'method': 'extstudy/refcheck.py evaluate(): mark inside the annotated complex of the same sound widened by 20 ms (s1_ok, s2_ok); within 100 ms of its centre (s1_wide, s2_wide); valid beats of training-a records outside annotated noise; record-clustered bootstrap 95% CI (2000 draws, seed 42)', 'status': 'REPORTED, NOT REQUIRED (EXP2_PREREG.md)', 'annotation_files': 409, 'tol_ms': 20.0, 'wide_ms': 100.0, 'acoustic': {'version': 'labels_acoustic_v1', 'beats': 5879, 'records': 242, 's1_ok': [0.9967681578499745, 0.9917068808750564, 1.0], 's2_ok': [0.99659806089471, 0.9916516113310854, 0.9998230197865107], 's1_wide': [0.9967681578499745, 0.9917068808750564, 1.0], 's2_wide': [0.9974485456710325, 0.9937806171311, 1.0], 's1_offset_ms': {'median': -24.74389160054713, 'p25': -34.979411284103534, 'p75': -12.066988394475953, 'abs_p95': 46.87498723166693}, 's2_offset_ms': {'median': -25.771630436757675, 'p25': -33.3958633214384, 'p75': -18.649965773830157, 'abs_p95': 46.05152530886609}}, 'ecg_v3_reference': {'version': 'labels_v3', 'beats': 5805, 'records': 240, 's1_ok': [0.9998277347114557, 0.9994506824267491, 1.0], 's2_ok': [0.9987941429801895, 0.99761493047405, 0.9996594135758854], 's1_wide': [1.0, 1.0, 1.0], 's2_wide': [0.9993109388458226, 0.9985802196151069, 0.9998346574511346], 's1_offset_ms': {'median': -24.647506970332245, 'p25': -34.91772076584709, 'p75': -12.180209248409568, 'abs_p95': 46.60255849691346}, 's2_offset_ms': {'median': -25.98803879932543, 'p25': -33.88293662454656, 'p75': -18.731936314908992, 'abs_p95': 46.49463648107712}}}            # acoustic labels against the hand-corrected training-a annotations: reported, not required (plan)
DRIVE_STUDY = '/content/drive/MyDrive/JBHI-Revision/Extension_Study'
print('SMOKE', SMOKE, '| kinds', KINDS, '| seeds', SEEDS, '| acoustic label gate', LABEL_GATE.get('status'))

## Environment: Drive, GPU, determinism

In [ ]:
import os, sys, json, time, hashlib, base64, shutil, errno, datetime
os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'
import numpy as np, torch
from google.colab import drive
drive.mount('/content/drive')
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
torch.backends.cuda.matmul.allow_tf32 = False
torch.backends.cudnn.allow_tf32 = False
torch.use_deterministic_algorithms(True, warn_only=True)
DEV = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
GPU = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none'
print('GPU', GPU, '| CPUs', os.cpu_count(), '| torch', torch.__version__)

## Study code (embedded, SHA-256 checked)

In [ ]:
MOD_B64 = {"__init__.py": "IiIiZXh0c3R1ZHk6IGNvZGUgZm9yIHRoZSBFSEotREggZXh0ZW5zaW9uIGV4cGVyaW1lbnRzIGFuZCB0aGUgZmlyc3Qgc2xpY2Ugb2YgdGhlIFBoRCBwcm9ncmFtbWUuIiIiCg==", "config.py": "IiIiUGF0aHMsIGNvbnN0YW50cywgYW5kIHNlZWRzIGZvciB0aGUgZXh0ZW5zaW9uIHN0dWR5IChzZWUgLi4vQ09ERV9ERVNJR04ubWQgYW5kIC4uLy4uLzA4X0VYVEVOU0lPTl9ERVNJR04ubWQpLg0KDQpUaGUgc2FtZSBjb2RlIHJ1bnMgbG9jYWxseSAoV2luZG93cywgcmF3IGRhdGEgb24gRTopIGFuZCBvbiBHb29nbGUgQ29sYWIgKERyaXZlIG1vdW50ZWQgYXQgL2NvbnRlbnQvZHJpdmUpLg0KQ2lyQ29yIHBhdGhzIGFyZSBleHBvc2VkIG9ubHkgdGhyb3VnaCBgY2lyY29yX3BhdGhzKClgOyB0ZXN0cy90ZXN0X2ZpcmV3YWxsLnB5IGZhaWxzIGlmIGEgdHJhaW5pbmcgb3INCnNlbGVjdGlvbiBtb2R1bGUgcmVmZXJlbmNlcyB0aGVtICh0aGUgQ2lyQ29yIGZpcmV3YWxsKS4NCiIiIg0KZnJvbSBkYXRhY2xhc3NlcyBpbXBvcnQgZGF0YWNsYXNzLCBmaWVsZA0KZnJvbSBkYXRldGltZSBpbXBvcnQgZGF0ZXRpbWUsIHRpbWV6b25lLCB0aW1lZGVsdGENCmltcG9ydCBvcw0KaW1wb3J0IHN5cw0KDQpPTl9DT0xBQiA9IG9zLnBhdGguaXNkaXIoJy9jb250ZW50JykgYW5kICgnZ29vZ2xlLmNvbGFiJyBpbiBzeXMubW9kdWxlcyBvciBvcy5wYXRoLmlzZGlyKCcvY29udGVudC9kcml2ZScpKQ0KDQojIC0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLSBsb2NhbCBsYXlvdXQNClBIRF9MT0NBTCA9ICdFOi9OVVNUIE1TIFRoZXNpcy9QaEQgTGV2ZWwgUmVzZWFyY2gnDQpMT0NBTCA9IGRpY3QoDQogICAgcGhkPVBIRF9MT0NBTCwNCiAgICBwY2dlY2dfY29kZT1QSERfTE9DQUwgKyAnL2NvZGUnLA0KICAgIGhhcm1vbmlzZWQ9UEhEX0xPQ0FMICsgJy9kZXJpdmVkL2hhcm1vbmlzZWQnLA0KICAgIGF1ZGl0X3RhYmxlPVBIRF9MT0NBTCArICcvZGVyaXZlZC9hdWRpdC9hdWRpdF9yZWNvcmRzLmNzdicsDQogICAgYmVhdHM9UEhEX0xPQ0FMICsgJy9kZXJpdmVkL2JlYXRzX3YyJywgICAgICAgICAgICAgICAgICAgICAjIHRlYWNoZXIgUi1wZWFrcyAocXJzLnB5LCByZWxhYmVsLnB5KQ0KICAgIHN0dWR5PVBIRF9MT0NBTCArICcvZXh0ZW5zaW9uX3N0dWR5JywNCiAgICBydW5zPVBIRF9MT0NBTCArICcvZXh0ZW5zaW9uX3N0dWR5L3J1bnMnLA0KICAgIHIxX3J1bj0nRTovTlVTVCBNUyBUaGVzaXMvRmluYWwgRHJhZnQvU3VibWl0dGVkIEZpbGUvUmV2aXNpb24tUjEvbm90ZWJvb2tzL1Jlc3VsdHMvUjFfR2F0aW5nX0NhbXBhaWduLzIwMjYxMDAzXzEyMzYnLA0KICAgIHIxX2J1aWxkZXI9J0U6L05VU1QgTVMgVGhlc2lzL0ZpbmFsIERyYWZ0L1N1Ym1pdHRlZCBGaWxlL1JldmlzaW9uLVIxL25vdGVib29rcy9idWlsZF9nYXRpbmdfY2FtcGFpZ24ucHknLA0KICAgIGdlbmVyYXRvcj0nRTovTlVTVCBNUyBUaGVzaXMvRmluYWwgRHJhZnQvU3VibWlzc2lvbi9tYXRlcmlhbHMvY29kZS93ZWlnaHRzL2dlbmVyYXRvci5wdCcsDQogICAgc3ByaW5nZXJfc2FtcGxlMjAxNj1QSERfTE9DQUwgKyAnL0RhdGFzZXRzL0RhdGFzZXQtNC9EYXRhc2V0LUNpbmMyMDE2L2NsYXNzaWZpY2F0aW9uLW9mLWhlYXJ0LXNvdW5kLXJlY29yZGluZ3MtdGhlLXBoeXNpb25ldC1jb21wdXRpbmctaW4tY2FyZGlvbG9neS1jaGFsbGVuZ2UtMjAxNi0xLjAuMCcsDQopDQpfQ0lSQ09SX0xPQ0FMID0gZGljdCgNCiAgICB0cmFpbmluZ19kYXRhPSdFOi9OVVNUIE1TIFRoZXNpcy9NaWxlc3RvbmUtMy9EYXRhc2V0L3RoZS1jaXJjb3ItZGlnaXNjb3BlLXBob25vY2FyZGlvZ3JhbS1kYXRhc2V0LTEuMC4zL3RyYWluaW5nX2RhdGEnLA0KICAgIG1ldGFkYXRhPSdFOi9OVVNUIE1TIFRoZXNpcy9NaWxlc3RvbmUtMy9EYXRhc2V0L3RoZS1jaXJjb3ItZGlnaXNjb3BlLXBob25vY2FyZGlvZ3JhbS1kYXRhc2V0LTEuMC4zL3RyYWluaW5nX2RhdGEuY3N2JywNCikNCg0KIyBhY291c3RpYyBsYWJlbHMgYW5kIHRoZSB0cmFpbmluZyBwYWNrIGJ1aWx0IGZyb20gdGhlbTsgdjEgKGxhYmVsc192MSwgcGFpcmVkX3YxKSBmYWlsZWQgdGhlIDIwMC1iZWF0IGxhYmVsIGNoZWNrDQojIG9uIDE3IE9jdCAyMDI2IGFuZCBzdGF5cyBvbiBkaXNrIHVuY2hhbmdlZCBhcyBpdHMgcmVjb3JkIChwcmVyZWcvQ0hBTkdFU19BRlRFUl9GUkVFWkUubWQpDQojIHYyIChsYWJlbHNfdjIsIHBhaXJlZF92MikgZmFpbGVkIHRoZSBibGluZGVkIHJlLWNoZWNrIG9uIDE4IE9jdCAyMDI2IGFuZCBzdGF5cyBvbiBkaXNrIHVuY2hhbmdlZA0KTEFCRUxTID0gJ2xhYmVsc192MycNClBBQ0sgPSAncGFpcmVkX3YzJw0KDQojIC0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLSBEcml2ZSBsYXlvdXQgKGxhdGVzdCwgZnJvbSB0aGUgUjEgbm90ZWJvb2tzKQ0KRFJJVkVfUk9PVCA9ICcvY29udGVudC9kcml2ZS9NeURyaXZlJw0KRFJJVkUgPSBkaWN0KA0KICAgIHJvb3Q9RFJJVkVfUk9PVCwNCiAgICBzdHVkeT1EUklWRV9ST09UICsgJy9KQkhJLVJldmlzaW9uL0V4dGVuc2lvbl9TdHVkeScsICAgICAgICAgICAgICAgICAjIG5ldzogZXZlcnl0aGluZyB0aGlzIHN0dWR5IHdyaXRlcw0KICAgIGNvcnB1cz1EUklWRV9ST09UICsgJy9KQkhJLVJldmlzaW9uL0V4dGVuc2lvbl9TdHVkeS9jb3JwdXMnLCAgICAgICAgICAjIHRoZSB0cmFpbmluZyBwYWNrIGJ1aWx0IGxvY2FsbHksIHVwbG9hZGVkDQogICAgY2hlY2twb2ludHM9RFJJVkVfUk9PVCArICcvSkJISS1SZXZpc2lvbi9FeHRlbnNpb25fU3R1ZHkvY2hlY2twb2ludHMnLA0KICAgIHJ1bnM9RFJJVkVfUk9PVCArICcvSkJISS1SZXZpc2lvbi9FeHRlbnNpb25fU3R1ZHkvcnVucycsDQogICAgcGFpcmVkX3Jhdz1EUklWRV9ST09UICsgJy9KQkhJLVJldmlzaW9uL0V4dGVuc2lvbl9TdHVkeS9wYWlyZWRfZGF0YXNldHMnLCAgICMgb3B0aW9uYWwgcmF3IHVwbG9hZCAobm90IG5lZWRlZCBmb3IgdHJhaW5pbmcpDQogICAgcjFfcm9vdD1EUklWRV9ST09UICsgJy9KQkhJLVJldmlzaW9uL1IxX0dhdGluZ19DYW1wYWlnbicsDQogICAgZ2VuZXJhdG9yPURSSVZFX1JPT1QgKyAnL09jdG9iZXIgVGhlc2lzIERhdGFzZXQvdHJhaWw0X2NoZWNrcG9pbnRzL3RyYWlsNF9iZXN0LnB0JywNCikNCl9DSVJDT1JfRFJJVkVfSElOVCA9IChEUklWRV9ST09UICsgJy9KQkhJLVJldmlzaW9uL09jdG9iZXIgVGhlc2lzIERhdGFzZXQvQ2lyQ29yIERpZ2lTY29wZS8nDQogICAgICAgICAgICAgICAgICAgICAgJ3RoZS1jaXJjb3ItZGlnaXNjb3BlLXBob25vY2FyZGlvZ3JhbS1kYXRhc2V0LTEuMC4zL3RoZS1jaXJjb3ItZGlnaXNjb3BlLXBob25vY2FyZGlvZ3JhbS1kYXRhc2V0LTEuMC4zL3RyYWluaW5nX2RhdGEnKQ0KDQoNCmRlZiBwYXRocygpOg0KICAgICIiIlRoZSBhY3RpdmUgcGF0aCBzZXQgZm9yIHRoaXMgZW52aXJvbm1lbnQgKENpckNvciBleGNsdWRlZCBvbiBwdXJwb3NlKS4iIiINCiAgICByZXR1cm4gRFJJVkUgaWYgT05fQ09MQUIgZWxzZSBMT0NBTA0KDQoNCmRlZiBjaXJjb3JfcGF0aHMoKToNCiAgICAiIiJDaXJDb3IgbG9jYXRpb25zLiBBbGxvd2VkIGNhbGxlcnM6IGluZmVyX2NpcmNvciwgc3ByaW5nZXIsIHNwbGl0cyAobWV0YWRhdGEgb25seSksIHRoZSBSMSBhZGFwdGVyLCBzdGF0cy4iIiINCiAgICBpZiBub3QgT05fQ09MQUI6DQogICAgICAgIHJldHVybiBkaWN0KF9DSVJDT1JfTE9DQUwpDQogICAgdGQgPSBfQ0lSQ09SX0RSSVZFX0hJTlQgaWYgb3MucGF0aC5pc2RpcihfQ0lSQ09SX0RSSVZFX0hJTlQpIGVsc2UgX2Rpc2NvdmVyX2NpcmNvcl9kcml2ZSgpDQogICAgcmV0dXJuIGRpY3QodHJhaW5pbmdfZGF0YT10ZCwgbWV0YWRhdGE9b3MucGF0aC5qb2luKG9zLnBhdGguZGlybmFtZSh0ZCksICd0cmFpbmluZ19kYXRhLmNzdicpKQ0KDQoNCmRlZiBfZGlzY292ZXJfY2lyY29yX2RyaXZlKG1heGRlcHRoPTgpOg0KICAgICIiIlNhbWUgcnVsZSBhcyB0aGUgUjEgbm90ZWJvb2s6IHRoZSB0cmFpbmluZ19kYXRhIGZvbGRlciB3aXRoIHRoZSBtb3N0IHBhaXJlZCAud2F2ICsgLnRzdiBmaWxlcy4iIiINCiAgICBiZXN0LCBiZXN0X24gPSBOb25lLCAtMQ0KICAgIHJvb3QgPSBvcy5wYXRoLmFic3BhdGgoRFJJVkVfUk9PVCkNCiAgICBmb3IgZCwgc3ViZGlycywgZmlsZXMgaW4gb3Mud2Fsayhyb290KToNCiAgICAgICAgaWYgZFtsZW4ocm9vdCk6XS5jb3VudChvcy5zZXApID49IG1heGRlcHRoOg0KICAgICAgICAgICAgc3ViZGlyc1s6XSA9IFtdDQogICAgICAgICAgICBjb250aW51ZQ0KICAgICAgICBzdWJkaXJzWzpdID0gW3MgZm9yIHMgaW4gc3ViZGlycyBpZiBzIG5vdCBpbiB7Jy5UcmFzaCcsICcuaXB5bmJfY2hlY2twb2ludHMnLCAnX19weWNhY2hlX18nfV0NCiAgICAgICAgaWYgb3MucGF0aC5iYXNlbmFtZShkKSA9PSAndHJhaW5pbmdfZGF0YSc6DQogICAgICAgICAgICB3YXYgPSB7Zls6LTRdIGZvciBmIGluIGZpbGVzIGlmIGYuZW5kc3dpdGgoJy53YXYnKX0NCiAgICAgICAgICAgIG4gPSBzdW0oZls6LTRdIGluIHdhdiBmb3IgZiBpbiBmaWxlcyBpZiBmLmVuZHN3aXRoKCcudHN2JykpDQogICAgICAgICAgICBpZiBuID4gYmVzdF9uOg0KICAgICAgICAgICAgICAgIGJlc3QsIGJlc3RfbiA9IGQsIG4NCiAgICBpZiBiZXN0IGlzIE5vbmU6DQogICAgICAgIHJhaXNlIEZpbGVOb3RGb3VuZEVycm9yKCdubyBDaXJDb3IgdHJhaW5pbmdfZGF0YSBmb2xkZXIgb24gRHJpdmUnKQ0KICAgIHJldHVybiBiZXN0DQoNCg0KIyAtLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0gY29uc3RhbnRzIChmaXhlZCBieSAwOF9FWFRFTlNJT05fREVTSUdOLm1kKQ0KQGRhdGFjbGFzcyhmcm96ZW49VHJ1ZSkNCmNsYXNzIFNpZ25hbDoNCiAgICBmc19lY2c6IGludCA9IDUwMA0KICAgIGZzX3BjZ19oYXJtb25pc2VkOiBpbnQgPSAyMDAwDQogICAgZnNfcGNnX21vZGVsOiBpbnQgPSAxMDAwDQogICAgZnNfZnJhbWVzOiBpbnQgPSAyNTAgICAgICAgICAgICAgICAgICAgICMgNCBtcyBob3ANCiAgICBiYW5kX2h6OiB0dXBsZSA9ICgyNSwgNDAwKQ0KICAgIHN0ZnRfd2luX21zOiBpbnQgPSAzMg0KICAgIHN0ZnRfYmluczogaW50ID0gMjQgICAgICAgICAgICAgICAgICAgICAjIDMxIHRvIDM5MCBIeg0KICAgIGZzX2NpcmNvcjogaW50ID0gNDAwMA0KDQoNCkBkYXRhY2xhc3MoZnJvemVuPVRydWUpDQpjbGFzcyBGb2xkczoNCiAgICBuX3NwbGl0czogaW50ID0gMTANCiAgICBzZWVkOiBpbnQgPSA0Mg0KICAgIG1scF9zZWVkX2Jhc2VzOiB0dXBsZSA9ICg0MiwgMTA0MiwgMjA0MiwgMzA0MiwgNDA0MikNCiAgICBib290c3RyYXBfZHJhd3M6IGludCA9IDIwMDANCg0KDQpAZGF0YWNsYXNzKGZyb3plbj1UcnVlKQ0KY2xhc3MgQW5hbHlzaXM6DQogICAgYWxwaGFfdG90YWxfb25lX3NpZGVkOiBmbG9hdCA9IDAuMDI1DQogICAgbmlfbWFyZ2luX3BvaW50czogZmxvYXQgPSAyLjUNCiAgICBmYWxsYmFja19mYWlsX3NoYXJlOiBmbG9hdCA9IDAuMDUNCiAgICBnYXRlX2JfdG9sZXJhbmNlc19tczogdHVwbGUgPSAoNDAsIDYwLCAxMDApDQoNCg0KUEtUID0gdGltZXpvbmUodGltZWRlbHRhKGhvdXJzPTUpKQ0KRlJFRVpFX1QgPSBkYXRldGltZSgyMDI2LCAxMCwgMzAsIDIzLCA1OSwgdHppbmZvPVBLVCkgICAgICAgIyBUIGFuZCBXIHdlaWdodHMgZnJvemVuDQpGUkVFWkVfU19SVCA9IGRhdGV0aW1lKDIwMjYsIDEwLCAyOSwgMjMsIDU5LCB0emluZm89UEtUKSAgICAjIHJldHJhaW5lZCBTcHJpbmdlcjogZnJvemVuIG9yIGRyb3BwZWQNCkZSRUVaRV9BTEwgPSBkYXRldGltZSgyMDI2LCAxMSwgNSwgMjMsIDU5LCB0emluZm89UEtUKSAgICAgICMgcmVzdWx0cyBlbnRlciB0aGUgcGFwZXIgYXMgdGhleSBzdGFuZA0KRFVFID0gZGF0ZXRpbWUoMjAyNiwgMTEsIDIwLCAyMywgNTksIHR6aW5mbz1QS1QpICAgICAgICAgICAgIyBFSEotREggcmV2aXNpb24gZHVlIChlZGl0b3IgY29uZmlybWVkIDE2IE9jdCAyMDI2KQ0KDQpAZGF0YWNsYXNzKGZyb3plbj1UcnVlKQ0KY2xhc3MgQmVhdHM6DQogICAgIiIiVGVhY2hlciBSLXBlYWsgcnVsZSAoUGFuZWwgSCwgMTUgT2N0IDIwMjYpLiIiIg0KICAgIHRvbF9tczogaW50ID0gNTAgICAgICAgICAgICAgICAgICAgICAgICAjIHZvdGUgYW5kIGFncmVlbWVudCB0b2xlcmFuY2UsIGFmdGVyIGVhY2ggZGV0ZWN0b3IncyByZWNvcmQgb2Zmc2V0DQogICAgb2Zmc2V0X3NlYXJjaF9tczogaW50ID0gMTAwICAgICAgICAgICAgICMgcGFpcnMgdXNlZCB0byBlc3RpbWF0ZSBhIGRldGVjdG9yJ3MgcmVjb3JkIG9mZnNldA0KICAgIHdmZGJfY2xpcDogZmxvYXQgPSAzLjAgICAgICAgICAgICAgICAgICAjIFhRUlMvR1FSUyBpbnB1dCBjbGlwcGVkIGF0IHRoaXMgbWFueSBRUlMgYW1wbGl0dWRlcw0KICAgIHhnX2YxX21pbjogZmxvYXQgPSAwLjkwICAgICAgICAgICAgICAgICAjIFhRUlMgYWdhaW5zdCBHUVJTDQogICAgcnhfZjFfbWluOiBmbG9hdCA9IDAuOTUgICAgICAgICAgICAgICAgICMgcm9idXN0IGRldGVjdG9yIGFnYWluc3QgWFFSUw0KICAgIGxvY2FsX3M6IGZsb2F0ID0gNS4wICAgICAgICAgICAgICAgICAgICAjIGNvbmZpcm1hdGlvbiBzaGFyZSBtZWFzdXJlZCBvdmVyIHJvYnVzdCBiZWF0cyB3aXRoaW4gKy8tIGxvY2FsX3MNCiAgICBsb2NhbF9jb25maXJtX21pbjogZmxvYXQgPSAwLjgwICAgICAgICAgIyBiZWxvdyB0aGlzIHNoYXJlIG9mIGdvb2Qgcm9idXN0IGJlYXRzLCB0aGUgbmVpZ2hib3VyaG9vZCBjYXJyaWVzIG5vIGxhYmVscw0KICAgIGV4dHJhX2FtcF9taW46IGZsb2F0ID0gMC41MCAgICAgICAgICAgICAjIHJlamVjdGVkIGNhbmRpZGF0ZSB3aXRoIHRoaXMgc2hhcmUgb2YgdGhlIGxvY2FsIGJlYXQgbGV2ZWwgKyBvbmUgd2ZkYg0KICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAjIGRldGVjdGlvbjogaXRzIGN5Y2xlIGNhcnJpZXMgbm8gbGFiZWxzIChoaWRkZW4gcHJlbWF0dXJlIGJlYXQpDQogICAgY29ycl9taW46IGZsb2F0ID0gMC44MCAgICAgICAgICAgICAgICAgICMgdGVtcGxhdGUgY29ycmVsYXRpb24gb2YgYSBnb29kIGJlYXQgKHJlYWwgRUNHOiBtZWRpYW4gcmVjb3JkIDAlIGJlbG93Ow0KICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAjIGJlYXRzIHBsYWNlZCBpbiBlbGVjdHJvZGUgbm9pc2U6IG1lZGlhbiAwLjUpDQogICAgd2luZG93X3M6IGZsb2F0ID0gNjAuMCAgICAgICAgICAgICAgICAgICMgcmVwb3J0aW5nIHdpbmRvd3MgKGFncmVlbWVudCBwZXIgbWludXRlKTsgdGhlIHN5bmMgd2luZG93cyBsaXZlIGluIHRoZSBhdWRpdA0KICAgIGdhcF9yYXRpbzogZmxvYXQgPSAxLjUgICAgICAgICAgICAgICAgICAjIDA4IGRlc2lnbjogUlIgYWJvdmUgMS41IHggdGhlIGxvY2FsIG1lZGlhbiBjYXJyaWVzIG5vIGxhYmVscw0KICAgIHJlZnJhY3RvcnlfczogZmxvYXQgPSAwLjIwICAgICAgICAgICAgICAjIHNoYXJlZCBkZWNvZGVyOiBtYXgoMC4yMCBzLCAwLjM1IHggZXhwZWN0ZWQgUlIpDQogICAgcmVmcmFjdG9yeV9yZWw6IGZsb2F0ID0gMC4zNQ0KICAgIGxhZ19nYXBfbWF4X21zOiBmbG9hdCA9IDguMCAgICAgICAgICAgICAjIFMtZG9taW5hbnQgbWludXMgUi1kb21pbmFudCBTMSBsYWcsIHBlciBkYXRhc2V0DQogICAgbWluX3NfYmVhdHM6IGludCA9IDUgICAgICAgICAgICAgICAgICAgICMgUy1kb21pbmFudCBiZWF0cyBhIHJlY29yZCBuZWVkcyB0byBlbnRlciB0aGUgbGFnLWdhcCBjaGVjaw0KDQoNClNJR05BTCwgRk9MRFMsIEFOQUxZU0lTLCBCRUFUUyA9IFNpZ25hbCgpLCBGb2xkcygpLCBBbmFseXNpcygpLCBCZWF0cygpDQoNCiMgRVBITk9HUkFNIHJlY29yZHMgdGhhdCBkdXBsaWNhdGUgb3RoZXIgcmVjb3JkcyBieXRlIGZvciBieXRlIChQYW5lbCBBLCBEMjsgdmVyaWZpZWQgMTUgT2N0IDIwMjYpDQpFWENMVURFX1JFQ09SRFMgPSB7KCdlcGhub2dyYW0nLCAnRUNHUENHMDAwMScpLCAoJ2VwaG5vZ3JhbScsICdFQ0dQQ0cwMDAyJyl9DQojIG9uZSBFQ0cgbGVhZCBwZXIgZGF0YXNldCAoUGFuZWwgQSwgRDMpOyBmYWxsIGJhY2sgb25seSB3aGVuIHRoaXMgbGVhZCBmYWlscyB0aGUgRUNHIGNoZWNrDQpQUklNQVJZX0xFQUQgPSB7J3NlbnNzbWFydCc6ICdJSScsICdic3NsYWInOiAnZWNnX2NoNCd9DQpMRUFEX09SREVSID0geydzZW5zc21hcnQnOiAoJ0lJJywgJ0knLCAnVjMnLCAnVjQnKSwgJ2Jzc2xhYic6ICgnZWNnX2NoNCcsICdlY2dfY2g1JywgJ2VjZ19jaDYnLCAnZWNnX2NoNycpfQ0K", "frontend.py": "IiIiTW9kZWwgaW5wdXQ6IGxvZy1wb3dlciBzcGVjdHJvZ3JhbSBvZiB0aGUgUENHICgwOCBkZXNpZ24sIHNlY3Rpb24gMi43KS4KClBDRyBhdCAxIGtIeiwgYmFuZC1wYXNzIDI1IHRvIDQwMCBIeiAoc2Vjb25kLW9yZGVyIHNlY3Rpb25zLCBwb2xlIGFzc2VydGlvbikgYXBwbGllZCBvbmNlIHRvIHRoZSB3aG9sZSByZWNvcmRpbmc7CmZyYW1lcyBvZiAzMiBzYW1wbGVzICgzMiBtcyBIYW5uKSBldmVyeSA0IHNhbXBsZXMgKDI1MCBIeiksIHplcm8tcGFkZGVkIHRvIDY0IHBvaW50cywgYmlucyAyIHRvIDI1ICgzMS4yNSB0byAzOTAuNiBIeiwKMjQgYmlucyk7IGxvZyBwb3dlcjsgZWFjaCBiaW4ncyBtZWRpYW4gb3ZlciB0aGUgd2hvbGUgcmVjb3JkaW5nIHN1YnRyYWN0ZWQuIEZyYW1lIGogY292ZXJzIHNhbXBsZXMgWzRqLCA0aiArIDMyKQphbmQgaXMgY2VudHJlZCBhdCAoNGogKyAxNikgLyAxMDAwIHMuIEZyYW1lcyB1c2Ugb25seSBzYW1wbGVzIGluc2lkZSB0aGVpciB3aW5kb3csIHNvIGEgY3JvcCB0aGF0IHN0YXJ0cyBvbiB0aGUgNC1zYW1wbGUKZ3JpZCB5aWVsZHMgZXhhY3RseSB0aGUgZnJhbWVzIG9mIHRoZSB3aG9sZSByZWNvcmRpbmcuIFRoZSBudW1weSB2ZXJzaW9uIGJ1aWxkcyB0aGUgY29ycHVzOyB0aGUgdG9yY2ggdmVyc2lvbiBydW5zCm9uIENvbGFiLCBhbmQgYSB0ZXN0IGhvbGRzIHRoZW0gZXF1YWwuCiIiIgppbXBvcnQgbnVtcHkgYXMgbnAKaW1wb3J0IHNjaXB5LnNpZ25hbCBhcyBzZwoKRlMgPSAxMDAwCldJTiwgSE9QLCBORkZUID0gMzIsIDQsIDY0CkJJTlMgPSBzbGljZSgyLCAyNikgICAgICAgICAgICAgICAgICAgICAgICAjIDMxLjI1IHRvIDM5MC42IEh6CkJBTkQgPSAoMjUuMCwgNDAwLjApCkVQUyA9IDFlLTEwCgoKZGVmIGJhbmRwYXNzX3Nvcyhmcz1GUyk6CiAgICBzb3MgPSBzZy5idXR0ZXIoNCwgQkFORCwgYnR5cGU9J2JhbmRwYXNzJywgZnM9ZnMsIG91dHB1dD0nc29zJykKICAgIGFzc2VydCBucC5hYnMoc2cuc29zMnpwayhzb3MpWzFdKS5tYXgoKSA8IDAuOTk5OSwgJ3Vuc3RhYmxlIGJhbmQtcGFzcycKICAgIHJldHVybiBzb3MKCgpkZWYgYmFuZHBhc3MocGNnLCBmcz1GUyk6CiAgICByZXR1cm4gbnAuYXNjb250aWd1b3VzYXJyYXkoc2cuc29zZmlsdGZpbHQoYmFuZHBhc3Nfc29zKGZzKSwgbnAuYXNhcnJheShwY2csIG5wLmZsb2F0NjQpKSkKCgpkZWYgdG9fMWsocGNnLCBmc19pbik6CiAgICBmcm9tIG1hdGggaW1wb3J0IGdjZAogICAgZnNfaW4gPSBpbnQocm91bmQoZnNfaW4pKQogICAgaWYgZnNfaW4gPT0gRlM6CiAgICAgICAgcmV0dXJuIG5wLmFzYXJyYXkocGNnLCBucC5mbG9hdDY0KQogICAgZyA9IGdjZChmc19pbiwgRlMpCiAgICByZXR1cm4gc2cucmVzYW1wbGVfcG9seShucC5hc2FycmF5KHBjZywgbnAuZmxvYXQ2NCksIEZTIC8vIGcsIGZzX2luIC8vIGcpCgoKZGVmIG5fZnJhbWVzKG5fc2FtcGxlcyk6CiAgICByZXR1cm4gbWF4KDAsIChuX3NhbXBsZXMgLSBXSU4pIC8vIEhPUCArIDEpCgoKZGVmIGZyYW1lX3RpbWVzX3Mobl9zYW1wbGVzLCBzdGFydF9zYW1wbGU9MCk6CiAgICByZXR1cm4gKHN0YXJ0X3NhbXBsZSArIG5wLmFyYW5nZShuX2ZyYW1lcyhuX3NhbXBsZXMpKSAqIEhPUCArIFdJTiAvIDIpIC8gRlMKCgpkZWYgbG9ncG93ZXIoeCk6CiAgICAiIiJCYW5kLXBhc3NlZCBQQ0cgKDEga0h6KSAtPiAoMjQsIGZyYW1lcykgbG9nIHBvd2VyLCBiZWZvcmUgbWVkaWFuIHN1YnRyYWN0aW9uLiIiIgogICAgeCA9IG5wLmFzYXJyYXkoeCwgbnAuZmxvYXQ2NCkKICAgIG5mID0gbl9mcmFtZXMobGVuKHgpKQogICAgaWYgbmYgPT0gMDoKICAgICAgICByZXR1cm4gbnAuemVyb3MoKEJJTlMuc3RvcCAtIEJJTlMuc3RhcnQsIDApKQogICAgaWR4ID0gbnAuYXJhbmdlKFdJTilbTm9uZSwgOl0gKyBIT1AgKiBucC5hcmFuZ2UobmYpWzosIE5vbmVdCiAgICBmciA9IHhbaWR4XSAqIG5wLmhhbm5pbmcoV0lOICsgMilbMTotMV1bTm9uZSwgOl0KICAgIHNwZWMgPSBucC5hYnMobnAuZmZ0LnJmZnQoZnIsIG49TkZGVCwgYXhpcz0xKSkgKiogMgogICAgcmV0dXJuIG5wLmxvZyhzcGVjWzosIEJJTlNdICsgRVBTKS5UCgoKZGVmIGJpbl9tZWRpYW5zKHhfYnApOgogICAgcmV0dXJuIG5wLm1lZGlhbihsb2dwb3dlcih4X2JwKSwgYXhpcz0xKQoKCmRlZiBsb2dzcGVjKHhfYnAsIG1lZGlhbnM9Tm9uZSk6CiAgICAiIiJNZWRpYW4tbm9ybWFsaXNlZCBzcGVjdHJvZ3JhbTsgbWVkaWFucyBmcm9tIHRoZSB3aG9sZSByZWNvcmRpbmcgKHBhc3MgdGhlbSBmb3IgY3JvcHMpLiIiIgogICAgTCA9IGxvZ3Bvd2VyKHhfYnApCiAgICBtID0gbnAubWVkaWFuKEwsIGF4aXM9MSkgaWYgbWVkaWFucyBpcyBOb25lIGVsc2UgbnAuYXNhcnJheShtZWRpYW5zKQogICAgcmV0dXJuIChMIC0gbVs6LCBOb25lXSkuYXN0eXBlKG5wLmZsb2F0MzIpCgoKZGVmIGxvZ3NwZWNfdG9yY2goeF9icCwgbWVkaWFucyk6CiAgICAiIiJUb3JjaCB0d2luIG9mIGxvZ3NwZWMgZm9yIGJhdGNoZXM6IHhfYnAgKEIsIG4pIGZsb2F0MzIsIG1lZGlhbnMgKEIsIDI0KS4gUmV0dXJucyAoQiwgMjQsIGZyYW1lcykuIiIiCiAgICBpbXBvcnQgdG9yY2gKICAgIHdpbiA9IHRvcmNoLmhhbm5fd2luZG93KFdJTiArIDIsIHBlcmlvZGljPUZhbHNlLCBkdHlwZT14X2JwLmR0eXBlLCBkZXZpY2U9eF9icC5kZXZpY2UpWzE6LTFdCiAgICBmciA9IHhfYnAudW5mb2xkKC0xLCBXSU4sIEhPUCkgKiB3aW4gICAgICAgICAgICAgICAgICAgICAgIyAoQiwgZnJhbWVzLCAzMikKICAgIHNwZWMgPSB0b3JjaC5mZnQucmZmdChmciwgbj1ORkZULCBkaW09LTEpLmFicygpICoqIDIKICAgIEwgPSB0b3JjaC5sb2coc3BlY1suLi4sIEJJTlNdICsgRVBTKS50cmFuc3Bvc2UoLTEsIC0yKSAgICAjIChCLCAyNCwgZnJhbWVzKQogICAgcmV0dXJuIEwgLSBtZWRpYW5zWy4uLiwgTm9uZV0K", "augment.py": "IiIiVHJhaW5pbmcgYXVnbWVudGF0aW9uIGZvciBUIGFuZCBXICgwOCBkZXNpZ24sIHNlY3Rpb24gMi43OyBwcmVyZWcgc2VjdGlvbiA3KS4gUGFpcmVkIGRhdGEgb25seS4KCkludGVydmFsIHdhcnBpbmcgKG9uIHNwZWN0cm9ncmFtIGZyYW1lcyk6IGEgcnVuIG9mIGNvbnNlY3V0aXZlIHZhbGlkIGJlYXRzIGlzIHJlLXRpbWVkIHRvIGEgdGFyZ2V0IGhlYXJ0IHJhdGUgaW4gNzAgdG8KMTgwIGJwbS4gRWFjaCBoZWFydCBzb3VuZCBrZWVwcyBhIHdpbmRvdyBvZiBwbHVzIG9yIG1pbnVzIDUwIG1zIGFyb3VuZCBpdHMgbGFiZWwgdW5jaGFuZ2VkOyBvbmx5IHRoZSBzeXN0b2xpYyBnYXAKKFMxIHdpbmRvdyBlbmQgdG8gUzIgd2luZG93IHN0YXJ0KSBhbmQgdGhlIGRpYXN0b2xpYyBnYXAgKFMyIHdpbmRvdyBlbmQgdG8gdGhlIG5leHQgUzEgd2luZG93IHN0YXJ0KSBhcmUgcmVzYW1wbGVkLgpUYXJnZXQgUlIgPSA2MCAvIEhSIHdpdGggYSBwZXItYmVhdCBqaXR0ZXIgKENWIDMlKTsgdGFyZ2V0IFMxLXRvLVMyIGludGVydmFsID0gdGhlIGJlYXQncyBvd24gaW50ZXJ2YWwgeApmKEhSX3RhcmdldCkgLyBmKEhSX3NvdXJjZSksIHdpdGggZiA9IGxhd19TMiAtIGxhd19TMSBvZiBpdHMgZGF0YXNldCAodGhlIHBhaXJlZCBsYXcsIDxjb25maWcuTEFCRUxTPi9sYXdzLmpzb24pLiBFdmVyeSBnYXAKY2hhbmdlIGlzIHJvdW5kZWQgdG8gd2hvbGUgZnJhbWVzLCBzbyB0aGUga2VwdCB3aW5kb3dzIG1vdmUgYnkgd2hvbGUgZnJhbWVzIGFuZCBhcmUgY29waWVkIGV4YWN0bHkuIExhYmVscywgY3ljbGUKYm91bmRzIGFuZCB0aGUgbG9zcyBtYXNrIG1vdmUgd2l0aCB0aGUgc2FtZSBwaWVjZXdpc2UtbGluZWFyIHRpbWUgbWFwLiBBIGJlYXQgd2hvc2UgdGFyZ2V0IGdhcHMgd291bGQgZmFsbCB1bmRlcgoxMCBtcyAoc3lzdG9sZSkgb3IgMjAgbXMgKGRpYXN0b2xlKSBsb3dlcnMgdGhlIHRhcmdldCByYXRlIGZvciB0aGUgd2hvbGUgcnVuIHRvIHRoZSBmYXN0ZXN0IGZlYXNpYmxlIG9uZS4KCldhdmVmb3JtIGF1Z21lbnRhdGlvbiAoYmVmb3JlIHRoZSBzcGVjdHJvZ3JhbSk6IGFkZGl0aXZlIG5vaXNlIGF0IDAgdG8gMjAgZEIgU05SOyBtdXJtdXItbGlrZSBiYW5kIG5vaXNlIGJldHdlZW4gUzEKYW5kIFMyIChwbGF0ZWF1LCBjcmVzY2VuZG8sIGRlY3Jlc2NlbmRvLCBvciBkaWFtb25kIGVudmVsb3BlKS4gU3BlY3Ryb2dyYW0gYXVnbWVudGF0aW9uOiBkZXZpY2UtcmVzcG9uc2UgdGlsdCBhY3Jvc3MKdGhlIDI0IGJpbnM7IGZyZXF1ZW5jeSBtYXNraW5nLgoiIiIKaW1wb3J0IG51bXB5IGFzIG5wCgpGUkFNRV9TID0gMC4wMDQKS0VFUF9TID0gMC4wNTAKTUlOX1NZU19HQVBfUywgTUlOX0RJQV9HQVBfUyA9IDAuMDEwLCAwLjAyMApIUl9SQU5HRSA9ICg3MC4wLCAxODAuMCkKCgpkZWYgbGF3X21zKGxhdywgZGF0YXNldCwgaHIpOgogICAgc2xvcGUgPSBsYXdbJ3Nsb3BlJ11bZGF0YXNldF0gaWYgaXNpbnN0YW5jZShsYXdbJ3Nsb3BlJ10sIGRpY3QpIGVsc2UgbGF3WydzbG9wZSddCiAgICByZXR1cm4gbGF3WydpbnRlcmNlcHQnXVtkYXRhc2V0XSArIHNsb3BlICogaHIKCgpkZWYgc3lzdG9sZV9tcyhsYXdzLCBkYXRhc2V0LCBocik6CiAgICAiIiJQYWlyZWQgbGF3IG9mIHRoZSBTMS10by1TMiBpbnRlcnZhbCAoZW52ZWxvcGUgcGVha3MpIGZvciBhIGRhdGFzZXQgYXQgYSBoZWFydCByYXRlLiIiIgogICAgcmV0dXJuIGxhd19tcyhsYXdzWydzMiddLCBkYXRhc2V0LCBocikgLSBsYXdfbXMobGF3c1snczEnXSwgZGF0YXNldCwgaHIpCgoKZGVmIF9rbm90cyhzMSwgczIsIHJyX3QsIHN5c190KToKICAgICIiIlNvdXJjZSBhbmQgdGFyZ2V0IGtub3QgdGltZXMgZm9yIGEgcnVuIG9mIGJlYXRzLiBzMSwgczI6IChrLCkgc291cmNlIGxhYmVsIHRpbWVzOyB0aGUgcnVuIGVuZHMgYXQgdGhlIFMxIHdpbmRvdwogICAgb2YgYmVhdCBrIChzMSBoYXMgayArIDEgZW50cmllcykuIHJyX3QsIHN5c190OiAoaywpIHRhcmdldCBSUiBhbmQgUzEtdG8tUzIgaW50ZXJ2YWxzLiBHYXAgY2hhbmdlcyBhcmUgcm91bmRlZCB0bwogICAgd2hvbGUgZnJhbWVzLiIiIgogICAgc3JjLCBkc3QgPSBbXSwgW10KICAgIHQgPSBzMVswXSAtIEtFRVBfUyAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICMgdGhlIGZpcnN0IHdpbmRvdyBzdGFydHMgd2hlcmUgaXQgc3RhcnRzIGluIHRoZSBzb3VyY2UKICAgIGZvciBpIGluIHJhbmdlKGxlbihzMikpOgogICAgICAgIGExLCBiMSA9IHMxW2ldIC0gS0VFUF9TLCBzMVtpXSArIEtFRVBfUwogICAgICAgIGEyLCBiMiA9IHMyW2ldIC0gS0VFUF9TLCBzMltpXSArIEtFRVBfUwogICAgICAgIGFfbmV4dCA9IHMxW2kgKyAxXSAtIEtFRVBfUwogICAgICAgIGdzX3NyYywgZ2Rfc3JjID0gYTIgLSBiMSwgYV9uZXh0IC0gYjIKICAgICAgICBnc190ID0gc3lzX3RbaV0gLSAyICogS0VFUF9TCiAgICAgICAgZ2RfdCA9IHJyX3RbaV0gLSBzeXNfdFtpXSAtIDIgKiBLRUVQX1MKICAgICAgICBnc190ID0gZ3Nfc3JjICsgbnAucm91bmQoKGdzX3QgLSBnc19zcmMpIC8gRlJBTUVfUykgKiBGUkFNRV9TCiAgICAgICAgZ2RfdCA9IGdkX3NyYyArIG5wLnJvdW5kKChnZF90IC0gZ2Rfc3JjKSAvIEZSQU1FX1MpICogRlJBTUVfUwogICAgICAgIHNyYyArPSBbYTEsIGIxLCBhMiwgYjJdCiAgICAgICAgZHN0ICs9IFt0LCB0ICsgMiAqIEtFRVBfUywgdCArIDIgKiBLRUVQX1MgKyBnc190LCB0ICsgNCAqIEtFRVBfUyArIGdzX3RdCiAgICAgICAgdCA9IHQgKyA0ICogS0VFUF9TICsgZ3NfdCArIGdkX3QKICAgIHNyYy5hcHBlbmQoczFbLTFdIC0gS0VFUF9TKTsgZHN0LmFwcGVuZCh0KQogICAgc3JjLmFwcGVuZChzMVstMV0gKyBLRUVQX1MpOyBkc3QuYXBwZW5kKHQgKyAyICogS0VFUF9TKQogICAgcmV0dXJuIG5wLmFycmF5KHNyYyksIG5wLmFycmF5KGRzdCkKCgpkZWYgZmVhc2libGVfaHIoczEsIHMyLCBocl90YXJnZXQsIHN5c19zcmMsIGZfc3JjLCBsYXdzLCBkYXRhc2V0KToKICAgICIiIkZhc3Rlc3QgcmF0ZSA8PSBocl90YXJnZXQgYXQgd2hpY2ggZXZlcnkgYmVhdCBrZWVwcyBpdHMgbWluaW11bSBnYXBzLiIiIgogICAgZm9yIGhyIGluIG5wLmFyYW5nZShocl90YXJnZXQsIEhSX1JBTkdFWzBdIC0gMWUtOSwgLTIuMCk6CiAgICAgICAgcnIgPSA2MC4wIC8gaHIKICAgICAgICBzeXNfdCA9IHN5c19zcmMgKiBzeXN0b2xlX21zKGxhd3MsIGRhdGFzZXQsIGhyKSAvIGZfc3JjCiAgICAgICAgaWYgbnAuYWxsKHN5c190IC0gMiAqIEtFRVBfUyA+PSBNSU5fU1lTX0dBUF9TKSBhbmQgbnAuYWxsKHJyIC0gc3lzX3QgLSAyICogS0VFUF9TID49IE1JTl9ESUFfR0FQX1MpOgogICAgICAgICAgICByZXR1cm4gZmxvYXQoaHIpCiAgICByZXR1cm4gZmxvYXQoSFJfUkFOR0VbMF0pCgoKZGVmIHdhcnBfbWFwKHMxLCBzMiwgaHJfc3JjLCBocl90YXJnZXQsIGxhd3MsIGRhdGFzZXQsIHJuZywgaml0dGVyX2N2PTAuMDMpOgogICAgIiIiUGllY2V3aXNlLWxpbmVhciB0aW1lIG1hcCBmb3IgYSBydW4gb2YgayBiZWF0czogcmV0dXJucyAoc3JjX2tub3RzLCBkc3Rfa25vdHMsIGhyX3VzZWQpLgogICAgczEgaGFzIGsgKyAxIGVudHJpZXMgKHRoZSBydW4ncyBsYXN0IFMxIGNsb3NlcyB0aGUgbGFzdCBjeWNsZSksIHMyIGhhcyBrIGVudHJpZXM7IGhyX3NyYzogKGssKSBsb2NhbCBzb3VyY2UgcmF0ZXMuIiIiCiAgICBzMSwgczIgPSBucC5hc2FycmF5KHMxLCBmbG9hdCksIG5wLmFzYXJyYXkoczIsIGZsb2F0KQogICAgc3lzX3NyYyA9IHMyIC0gczFbOi0xXQogICAgZl9zcmMgPSBzeXN0b2xlX21zKGxhd3MsIGRhdGFzZXQsIG5wLmFzYXJyYXkoaHJfc3JjLCBmbG9hdCkpIC8gMTAwMC4wCiAgICBociA9IGZlYXNpYmxlX2hyKHMxLCBzMiwgaHJfdGFyZ2V0LCBzeXNfc3JjLCBmX3NyYywgbGF3cywgZGF0YXNldCkKICAgIHJyX3QgPSA2MC4wIC8gaHIgKiAoMS4wICsgaml0dGVyX2N2ICogcm5nLnN0YW5kYXJkX25vcm1hbChsZW4oczIpKSkKICAgIHN5c190ID0gc3lzX3NyYyAqIChzeXN0b2xlX21zKGxhd3MsIGRhdGFzZXQsIGhyKSAvIDEwMDAuMCkgLyBmX3NyYwogICAgc3lzX3QgPSBucC5taW5pbXVtKHN5c190LCBycl90IC0gMiAqIEtFRVBfUyAtIE1JTl9ESUFfR0FQX1MpCiAgICBzeXNfdCA9IG5wLm1heGltdW0oc3lzX3QsIDIgKiBLRUVQX1MgKyBNSU5fU1lTX0dBUF9TKQogICAgc3JjLCBkc3QgPSBfa25vdHMoczEsIHMyLCBycl90LCBzeXNfdCkKICAgIHJldHVybiBzcmMsIGRzdCwgaHIKCgpkZWYgbWFwX3RpbWVzKHgsIHNyYywgZHN0KToKICAgICIiIlNvdXJjZSB0aW1lcyAtPiB0YXJnZXQgdGltZXM7IGxpbmVhciBiZXR3ZWVuIGtub3RzLCBzbG9wZSAxIG91dHNpZGUgdGhlIHJ1bi4iIiIKICAgIHggPSBucC5hc2FycmF5KHgsIGZsb2F0KQogICAgeSA9IG5wLmludGVycCh4LCBzcmMsIGRzdCkKICAgIHkgPSBucC53aGVyZSh4IDwgc3JjWzBdLCBkc3RbMF0gLSAoc3JjWzBdIC0geCksIHkpCiAgICByZXR1cm4gbnAud2hlcmUoeCA+IHNyY1stMV0sIGRzdFstMV0gKyAoeCAtIHNyY1stMV0pLCB5KQoKCmRlZiB3YXJwX2ZyYW1lcyhMLCB0MCwgc3JjLCBkc3QsIG5fb3V0LCBvdXRfdDApOgogICAgIiIiUmVzYW1wbGUgc3BlY3Ryb2dyYW0gZnJhbWVzIEwgKGJpbnMsIG4pIHdpdGggZnJhbWUgaiBhdCB0MCArIGogKiBGUkFNRV9TIG9udG8gbl9vdXQgdGFyZ2V0IGZyYW1lcyBzdGFydGluZyBhdAogICAgb3V0X3QwLCB0aHJvdWdoIHRoZSBpbnZlcnNlIG9mIHRoZSB0aW1lIG1hcCAobGluZWFyIGludGVycG9sYXRpb24gYmV0d2VlbiBzb3VyY2UgZnJhbWVzOyB3aG9sZS1mcmFtZSBzaGlmdHMgaW5zaWRlCiAgICB0aGUga2VwdCB3aW5kb3dzIGNvcHkgZnJhbWVzIGV4YWN0bHkpLiBUYXJnZXQgZnJhbWVzIG91dHNpZGUgdGhlIHJ1biBhcmUgemVybyAodGhlIG1lZGlhbiBsZXZlbCkgYW5kIHJlcG9ydGVkLiIiIgogICAgdHQgPSBvdXRfdDAgKyBucC5hcmFuZ2Uobl9vdXQpICogRlJBTUVfUwogICAgaW5zaWRlID0gKHR0ID49IGRzdFswXSkgJiAodHQgPD0gZHN0Wy0xXSkKICAgIHRzID0gbnAuaW50ZXJwKHR0LCBkc3QsIHNyYykgICAgICAgICAgICAgICAgICAgICAgICAgICMgaW52ZXJzZSBtYXAgKGRzdCBpcyBpbmNyZWFzaW5nKQogICAgcG9zID0gKHRzIC0gdDApIC8gRlJBTUVfUwogICAgcG9zX3IgPSBucC5yb3VuZChwb3MpCiAgICBwb3MgPSBucC53aGVyZShucC5hYnMocG9zIC0gcG9zX3IpIDwgMWUtNiwgcG9zX3IsIHBvcykgICAjIHdob2xlIGZyYW1lcyBzdGF5IHdob2xlIChubyBpbnRlcnBvbGF0aW9uIGRyaWZ0KQogICAgajAgPSBucC5jbGlwKG5wLmZsb29yKHBvcykuYXN0eXBlKGludCksIDAsIEwuc2hhcGVbMV0gLSAxKQogICAgajEgPSBucC5jbGlwKGowICsgMSwgMCwgTC5zaGFwZVsxXSAtIDEpCiAgICB3ID0gbnAuY2xpcChwb3MgLSBucC5mbG9vcihwb3MpLCAwLjAsIDEuMCkKICAgIG91dCA9IExbOiwgajBdICogKDEgLSB3KSArIExbOiwgajFdICogdwogICAgdmFsaWQgPSBpbnNpZGUgJiAocG9zID49IDApICYgKHBvcyA8PSBMLnNoYXBlWzFdIC0gMSkKICAgIG91dFs6LCB+dmFsaWRdID0gMC4wCiAgICByZXR1cm4gb3V0LmFzdHlwZShucC5mbG9hdDMyKSwgdmFsaWQKCgojIC0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLSB3YXZlZm9ybSBhbmQgc3BlY3Ryb2dyYW0gYXVnbWVudGF0aW9uCmRlZiBhZGRfbm9pc2UoeCwgcm5nLCBzbnJfZGJfcmFuZ2U9KDAuMCwgMjAuMCkpOgogICAgc25yID0gcm5nLnVuaWZvcm0oKnNucl9kYl9yYW5nZSkKICAgIHAgPSBucC5tZWFuKG5wLmFzYXJyYXkoeCwgbnAuZmxvYXQ2NCkgKiogMikgKyAxZS0xMgogICAgcmV0dXJuIHggKyBybmcuc3RhbmRhcmRfbm9ybWFsKGxlbih4KSkgKiBucC5zcXJ0KHAgLyAxMCAqKiAoc25yIC8gMTApKSwgZmxvYXQoc25yKQoKCmRlZiBtdXJtdXJfbm9pc2UoeCwgczEsIHMyLCBmcywgcm5nLCByZWxfZGJfcmFuZ2U9KC0yMC4wLCAwLjApKToKICAgICIiIkJhbmQtbGltaXRlZCBub2lzZSBiZXR3ZWVuIFMxIGFuZCBTMiBvZiBlYWNoIGJlYXQgKGVudmVsb3BlIHNoYXBlIGRyYXduIHBlciBjYWxsKSwgc2NhbGVkIHRvIHRoZSBTMSB3aW5kb3cgUk1TLiIiIgogICAgaW1wb3J0IHNjaXB5LnNpZ25hbCBhcyBzZwogICAgbG8gPSBybmcudW5pZm9ybSg2MCwgMjAwKTsgaGkgPSBtaW4obG8gKyBybmcudW5pZm9ybSgxMDAsIDI1MCksIDQyMCkKICAgIHNvcyA9IHNnLmJ1dHRlcig0LCBbbG8sIGhpXSwgYnR5cGU9J2JhbmRwYXNzJywgZnM9ZnMsIG91dHB1dD0nc29zJykKICAgIHNoYXBlID0gcm5nLmNob2ljZShbJ3BsYXRlYXUnLCAnY3Jlc2NlbmRvJywgJ2RlY3Jlc2NlbmRvJywgJ2RpYW1vbmQnXSkKICAgIHJlbCA9IDEwICoqIChybmcudW5pZm9ybSgqcmVsX2RiX3JhbmdlKSAvIDIwKQogICAgeSA9IG5wLmFycmF5KHgsIG5wLmZsb2F0NjQsIGNvcHk9VHJ1ZSkKICAgIGZvciBhLCBiIGluIHppcChzMSwgczIpOgogICAgICAgIGkwLCBpMSA9IGludCgoYSArIDAuMDMpICogZnMpLCBpbnQoKGIgLSAwLjAyKSAqIGZzKQogICAgICAgIGlmIGkxIC0gaTAgPCBpbnQoMC4wMyAqIGZzKSBvciBpMCA8IDAgb3IgaTEgPiBsZW4oeSk6CiAgICAgICAgICAgIGNvbnRpbnVlCiAgICAgICAgbiA9IGkxIC0gaTAKICAgICAgICB1ID0gbnAubGluc3BhY2UoMCwgMSwgbikKICAgICAgICBlbnYgPSB7J3BsYXRlYXUnOiBucC5vbmVzKG4pLCAnY3Jlc2NlbmRvJzogdSwgJ2RlY3Jlc2NlbmRvJzogMSAtIHUsICdkaWFtb25kJzogMSAtIG5wLmFicygyICogdSAtIDEpfVtzaGFwZV0KICAgICAgICBub2lzZSA9IHNnLnNvc2ZpbHQoc29zLCBybmcuc3RhbmRhcmRfbm9ybWFsKG4gKyAyMDApKVsyMDA6XQogICAgICAgIHcwLCB3MSA9IG1heCgwLCBpbnQoKGEgLSAwLjA1KSAqIGZzKSksIG1pbihsZW4oeSksIGludCgoYSArIDAuMDUpICogZnMpKQogICAgICAgIHJlZiA9IG5wLnNxcnQobnAubWVhbih5W3cwOncxXSAqKiAyKSkgKyAxZS0xMgogICAgICAgIHlbaTA6aTFdICs9IG5vaXNlIC8gKG5wLnN0ZChub2lzZSkgKyAxZS0xMikgKiByZWYgKiByZWwgKiBlbnYKICAgIHJldHVybiB5LCBkaWN0KGJhbmQ9KGZsb2F0KGxvKSwgZmxvYXQoaGkpKSwgc2hhcGU9c3RyKHNoYXBlKSwgcmVsPWZsb2F0KHJlbCkpCgoKZGVmIGRldmljZV90aWx0KEwsIHJuZywgbWF4X3RpbHQ9MS4wKToKICAgICIiIkxpbmVhciB0aWx0IGFjcm9zcyB0aGUgYmlucyBpbiBsb2ctcG93ZXIgdW5pdHMgKHBsdXMgb3IgbWludXMgbWF4X3RpbHQgYXQgdGhlIGJhbmQgZWRnZXMpLiIiIgogICAgYSA9IHJuZy51bmlmb3JtKC1tYXhfdGlsdCwgbWF4X3RpbHQpCiAgICByZXR1cm4gKEwgKyBhICogbnAubGluc3BhY2UoLTEsIDEsIEwuc2hhcGVbMF0pWzosIE5vbmVdKS5hc3R5cGUobnAuZmxvYXQzMiksIGZsb2F0KGEpCgoKZGVmIGZyZXFfbWFzayhMLCBybmcsIG1heF93aWR0aD0zKToKICAgIHcgPSBpbnQocm5nLmludGVnZXJzKDEsIG1heF93aWR0aCArIDEpKTsgcyA9IGludChybmcuaW50ZWdlcnMoMCwgTC5zaGFwZVswXSAtIHcgKyAxKSkKICAgIEwgPSBMLmNvcHkoKTsgTFtzOnMgKyB3XSA9IDAuMAogICAgcmV0dXJuIEwsIChzLCB3KQo=", "losses.py": "IiIiTG9zc2VzIGZvciBUIGFuZCBXICgwOCBkZXNpZ24sIHNlY3Rpb24gMi43OyBwcmVyZWcgc2VjdGlvbiA3KS4KClQ6IHBlbmFsdHktcmVkdWNlZCBmb2NhbCBsb3NzIChDZW50ZXJOZXQ6IGFscGhhIDIsIGJldGEgNCkgb24gR2F1c3NpYW4gZXZlbnQgdGFyZ2V0cyBhdCB0aGUgZnJhbWUgcmF0ZSAoc2lnbWEgMTIgbXMKZm9yIFMxIGFuZCBTMiwgOCBtcyBmb3IgUiksIGhlYWQgd2VpZ2h0cyAxLCAxLCAwLjUsIG5vcm1hbGlzZWQgYnkgdGhlIG51bWJlciBvZiBldmVudHMsIG1hc2tlZCBmcmFtZXMgY29udHJpYnV0aW5nCnplcm8gKGludmFsaWQgY3ljbGVzIGFyZSBtYXNrZWQsIG5ldmVyIGJhY2tncm91bmQpLgpXOiBMMSBvbiB0aGUgRUNHIGFuZCBvbiBpdHMgZmlyc3QgZGlmZmVyZW5jZSwgbWFza2VkIHRoZSBzYW1lIHdheS4KVGFyZ2V0cyBhcmUgYnVpbHQgZnJvbSBldmVudCB0aW1lcyBpbiBzZWNvbmRzOyBmcmFtZSBqIGlzIGNlbnRyZWQgYXQgZnJhbWVfdDAgKyBqIC8gMjUwIHMuCiIiIgppbXBvcnQgbnVtcHkgYXMgbnAKaW1wb3J0IHRvcmNoCgpGUkFNRV9IWiA9IDI1MC4wClNJR01BX1MgPSB7J3MxJzogMC4wMTIsICdzMic6IDAuMDEyLCAncic6IDAuMDA4fQpIRUFEX1dFSUdIVCA9IHsnczEnOiAxLjAsICdzMic6IDEuMCwgJ3InOiAwLjV9CkFMUEhBLCBCRVRBID0gMi4wLCA0LjAKCgpkZWYgZ2F1c3NpYW5fdGFyZ2V0cyhldmVudF9zLCBuX2ZyYW1lcywgZnJhbWVfdDAsIHNpZ21hX3MpOgogICAgIiIiKG5fZnJhbWVzLCkgaGVhdG1hcDogbWF4IG92ZXIgZXZlbnRzIG9mIGV4cCgtKHQgLSBlKV4yIC8gMiBzaWdtYV4yKSwgd2l0aCBleGFjdGx5IDEgYXQgZWFjaCBldmVudCdzIG5lYXJlc3QgZnJhbWUKICAgICh0aGUgQ2VudGVyTmV0IHBlYWspLCBhbmQgYSBib29sZWFuIGFycmF5IG1hcmtpbmcgdGhvc2UgcGVhayBmcmFtZXMuIiIiCiAgICB0ID0gZnJhbWVfdDAgKyBucC5hcmFuZ2Uobl9mcmFtZXMpIC8gRlJBTUVfSFoKICAgIHkgPSBucC56ZXJvcyhuX2ZyYW1lcywgbnAuZmxvYXQzMikKICAgIHBlYWsgPSBucC56ZXJvcyhuX2ZyYW1lcywgYm9vbCkKICAgIGZvciBlIGluIG5wLmFzYXJyYXkoZXZlbnRfcywgZmxvYXQpOgogICAgICAgIGlmIG5vdCBucC5pc2Zpbml0ZShlKSBvciBlIDwgdFswXSAtIDUgKiBzaWdtYV9zIG9yIGUgPiB0Wy0xXSArIDUgKiBzaWdtYV9zOgogICAgICAgICAgICBjb250aW51ZQogICAgICAgIGxvLCBoaSA9IG5wLnNlYXJjaHNvcnRlZCh0LCBlIC0gNSAqIHNpZ21hX3MpLCBucC5zZWFyY2hzb3J0ZWQodCwgZSArIDUgKiBzaWdtYV9zKQogICAgICAgIHlbbG86aGldID0gbnAubWF4aW11bSh5W2xvOmhpXSwgbnAuZXhwKC0wLjUgKiAoKHRbbG86aGldIC0gZSkgLyBzaWdtYV9zKSAqKiAyKSkKICAgICAgICBrID0gaW50KG5wLmNsaXAobnAucm91bmQoKGUgLSBmcmFtZV90MCkgKiBGUkFNRV9IWiksIDAsIG5fZnJhbWVzIC0gMSkpCiAgICAgICAgaWYgYWJzKHRba10gLSBlKSA8PSAwLjUgLyBGUkFNRV9IWiArIDFlLTk6CiAgICAgICAgICAgIHlba10sIHBlYWtba10gPSAxLjAsIFRydWUKICAgIHJldHVybiB5LCBwZWFrCgoKZGVmIGZvY2FsX2V2ZW50X2xvc3MobG9naXRzLCB0YXJnZXQsIHBlYWssIG1hc2spOgogICAgIiIibG9naXRzLCB0YXJnZXQgKGZsb2F0KSwgcGVhayAoYm9vbCksIG1hc2sgKGJvb2wpOiAoQiwgbikuIFBlbmFsdHktcmVkdWNlZCBmb2NhbCBsb3NzIHN1bW1lZCBvdmVyIHVubWFza2VkIGZyYW1lcywKICAgIGRpdmlkZWQgYnkgdGhlIG51bWJlciBvZiB1bm1hc2tlZCBwZWFrcyAoYXQgbGVhc3QgMSkuIiIiCiAgICBwID0gdG9yY2guc2lnbW9pZChsb2dpdHMpLmNsYW1wKDFlLTYsIDEgLSAxZS02KQogICAgbSA9IG1hc2suZmxvYXQoKQogICAgcG9zID0gLSgoMSAtIHApICoqIEFMUEhBKSAqIHRvcmNoLmxvZyhwKSAqIHBlYWsuZmxvYXQoKQogICAgbmVnID0gLSgoMSAtIHRhcmdldCkgKiogQkVUQSkgKiAocCAqKiBBTFBIQSkgKiB0b3JjaC5sb2coMSAtIHApICogKDEgLSBwZWFrLmZsb2F0KCkpCiAgICBuX3BvcyA9IChwZWFrLmZsb2F0KCkgKiBtKS5zdW0oKS5jbGFtcChtaW49MS4wKQogICAgcmV0dXJuICgocG9zICsgbmVnKSAqIG0pLnN1bSgpIC8gbl9wb3MKCgpkZWYgdF9sb3NzKGxvZ2l0cywgdGFyZ2V0cywgcGVha3MsIG1hc2ssIGhlYWRzPSgnczEnLCAnczInLCAncicpKToKICAgICIiImxvZ2l0cyAoQiwgbGVuKGhlYWRzKSwgbikgaW4gaGVhZCBvcmRlcjsgdGFyZ2V0cy9wZWFrcyBkaWN0cyBvZiAoQiwgbikgdGVuc29yczsgbWFzayAoQiwgbikuIFRoZSBhY291c3RpYy1sYWJlbAogICAgY29udHJvbCBhcm0gcGFzc2VzIGhlYWRzICgnczEnLCAnczInKS4iIiIKICAgIHRvdGFsID0gMC4wCiAgICBwYXJ0cyA9IHt9CiAgICBmb3IgaSwgaCBpbiBlbnVtZXJhdGUoaGVhZHMpOgogICAgICAgIHBhcnRzW2hdID0gZm9jYWxfZXZlbnRfbG9zcyhsb2dpdHNbOiwgaV0sIHRhcmdldHNbaF0sIHBlYWtzW2hdLCBtYXNrKQogICAgICAgIHRvdGFsID0gdG90YWwgKyBIRUFEX1dFSUdIVFtoXSAqIHBhcnRzW2hdCiAgICByZXR1cm4gdG90YWwsIHBhcnRzCgoKZGVmIHdfbG9zcyhwcmVkLCBlY2csIG1hc2spOgogICAgIiIicHJlZCwgZWNnLCBtYXNrOiAoQiwgbikuIE1hc2tlZCBMMSBvbiB0aGUgc2lnbmFsIHBsdXMgbWFza2VkIEwxIG9uIHRoZSBmaXJzdCBkaWZmZXJlbmNlLCBlYWNoIGEgbWVhbiBvdmVyIHVubWFza2VkCiAgICBmcmFtZXMuIiIiCiAgICBtID0gbWFzay5mbG9hdCgpCiAgICBsMSA9ICh0b3JjaC5hYnMocHJlZCAtIGVjZykgKiBtKS5zdW0oKSAvIG0uc3VtKCkuY2xhbXAobWluPTEuMCkKICAgIG1kID0gbVs6LCAxOl0gKiBtWzosIDotMV0KICAgIGwxZCA9ICh0b3JjaC5hYnModG9yY2guZGlmZihwcmVkLCBkaW09MSkgLSB0b3JjaC5kaWZmKGVjZywgZGltPTEpKSAqIG1kKS5zdW0oKSAvIG1kLnN1bSgpLmNsYW1wKG1pbj0xLjApCiAgICByZXR1cm4gbDEgKyBsMWQsIGRpY3QobDE9bDEsIGwxX2RpZmY9bDFkKQo=", "model.py": "IiIiMS1EIFUtTmV0IG9uIHRoZSBsb2ctcG93ZXIgc3BlY3Ryb2dyYW0gKDA4IGRlc2lnbiwgc2VjdGlvbiAyLjc7IHByZXJlZyBzZWN0aW9uIDcpLg0KDQpJbnB1dCAoQiwgMjQsIGZyYW1lcykgYXQgMjUwIEh6LiBFbmNvZGVyIDY0IC0+IDk2IC0+IDEyOCBjaGFubmVscyAodHdvIHN0cmlkZS0yIHN0ZXBzLCAzLXRhcCBjb252b2x1dGlvbnMpLA0KZWlnaHQgZGlsYXRlZCByZXNpZHVhbCBibG9ja3MgYXQgMTI4IGNoYW5uZWxzIChhIGRpbGF0ZWQgMy10YXAgY29udm9sdXRpb24gYW5kIGEgMXgxIGNvbnZvbHV0aW9uOyBkaWxhdGlvbnMgMSwgMiwgNCwgOCwNCjEsIDIsIDQsIDg7IHJlY2VwdGl2ZSBmaWVsZCBhYm91dCAxIHMpLCBkZWNvZGVyIGJhY2sgdG8gNjQgY2hhbm5lbHMgd2l0aCBza2lwIGNvbm5lY3Rpb25zLCBhbmQgMXgxIGhlYWRzIGF0IHRoZSBpbnB1dA0KZnJhbWUgcmF0ZS4gVDogdGhyZWUgZXZlbnQgbG9naXRzIChTMSwgUzIsIFIpLiBXOiBvbmUgRUNHIGNoYW5uZWwuIEdyb3VwTm9ybSAoaW5kZXBlbmRlbnQgb2YgYmF0Y2gNCmNvbXBvc2l0aW9uKSwgODcyLDM4NyBwYXJhbWV0ZXJzIHdpdGggdGhlIFQgaGVhZHMuIEFueSBmcmFtZSBjb3VudCB3b3JrczogdGhlIGlucHV0IGlzIHBhZGRlZCB0byBhIG11bHRpcGxlIG9mIDQNCmJ5IHJlZmxlY3Rpb24gYW5kIHRoZSBvdXRwdXQgY3JvcHBlZCBiYWNrLg0KIiIiDQppbXBvcnQgdG9yY2gNCmltcG9ydCB0b3JjaC5ubiBhcyBubg0KaW1wb3J0IHRvcmNoLm5uLmZ1bmN0aW9uYWwgYXMgRg0KDQpIRUFEUyA9IHsnVCc6ICgnczEnLCAnczInLCAncicpLCAnVyc6ICgnZWNnJywpLCAnQSc6ICgnczEnLCAnczInKX0gICAgICAgIyBBOiB0aGUgYWNvdXN0aWMtbGFiZWwgY29udHJvbCBhcm0gKEV4cGVyaW1lbnQgMiksIG5vIFIgaGVhZA0KDQoNCmRlZiBfbm9ybShjKToNCiAgICByZXR1cm4gbm4uR3JvdXBOb3JtKDgsIGMpDQoNCg0KY2xhc3MgQ29udkJsb2NrKG5uLk1vZHVsZSk6DQogICAgZGVmIF9faW5pdF9fKHNlbGYsIGNpbiwgY291dCwgaz0zKToNCiAgICAgICAgc3VwZXIoKS5fX2luaXRfXygpDQogICAgICAgIHNlbGYubmV0ID0gbm4uU2VxdWVudGlhbChubi5Db252MWQoY2luLCBjb3V0LCBrLCBwYWRkaW5nPWsgLy8gMiksIF9ub3JtKGNvdXQpLCBubi5HRUxVKCksDQogICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICBubi5Db252MWQoY291dCwgY291dCwgaywgcGFkZGluZz1rIC8vIDIpLCBfbm9ybShjb3V0KSwgbm4uR0VMVSgpKQ0KDQogICAgZGVmIGZvcndhcmQoc2VsZiwgeCk6DQogICAgICAgIHJldHVybiBzZWxmLm5ldCh4KQ0KDQoNCmNsYXNzIFJlc0Jsb2NrKG5uLk1vZHVsZSk6DQogICAgZGVmIF9faW5pdF9fKHNlbGYsIGMsIGRpbGF0aW9uLCBkcm9wb3V0PTAuMSk6DQogICAgICAgIHN1cGVyKCkuX19pbml0X18oKQ0KICAgICAgICBzZWxmLm5ldCA9IG5uLlNlcXVlbnRpYWwobm4uQ29udjFkKGMsIGMsIDMsIHBhZGRpbmc9ZGlsYXRpb24sIGRpbGF0aW9uPWRpbGF0aW9uKSwgX25vcm0oYyksIG5uLkdFTFUoKSwgbm4uRHJvcG91dChkcm9wb3V0KSwNCiAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgIG5uLkNvbnYxZChjLCBjLCAxKSwgX25vcm0oYykpDQoNCiAgICBkZWYgZm9yd2FyZChzZWxmLCB4KToNCiAgICAgICAgcmV0dXJuIEYuZ2VsdSh4ICsgc2VsZi5uZXQoeCkpDQoNCg0KY2xhc3MgVU5ldDFEKG5uLk1vZHVsZSk6DQogICAgZGVmIF9faW5pdF9fKHNlbGYsIGtpbmQ9J1QnLCBuX2luPTI0LCB3aWR0aHM9KDY0LCA5NiwgMTI4KSwgZGlsYXRpb25zPSgxLCAyLCA0LCA4LCAxLCAyLCA0LCA4KSk6DQogICAgICAgIHN1cGVyKCkuX19pbml0X18oKQ0KICAgICAgICBhc3NlcnQga2luZCBpbiBIRUFEUw0KICAgICAgICBzZWxmLmtpbmQgPSBraW5kDQogICAgICAgIHcwLCB3MSwgdzIgPSB3aWR0aHMNCiAgICAgICAgc2VsZi5lbmMwID0gQ29udkJsb2NrKG5faW4sIHcwKQ0KICAgICAgICBzZWxmLmRvd24xID0gbm4uQ29udjFkKHcwLCB3MSwgNCwgc3RyaWRlPTIsIHBhZGRpbmc9MSkNCiAgICAgICAgc2VsZi5lbmMxID0gQ29udkJsb2NrKHcxLCB3MSkNCiAgICAgICAgc2VsZi5kb3duMiA9IG5uLkNvbnYxZCh3MSwgdzIsIDQsIHN0cmlkZT0yLCBwYWRkaW5nPTEpDQogICAgICAgIHNlbGYubWlkID0gbm4uU2VxdWVudGlhbCgqW1Jlc0Jsb2NrKHcyLCBkKSBmb3IgZCBpbiBkaWxhdGlvbnNdKQ0KICAgICAgICBzZWxmLnVwMiA9IG5uLkNvbnZUcmFuc3Bvc2UxZCh3MiwgdzEsIDQsIHN0cmlkZT0yLCBwYWRkaW5nPTEpDQogICAgICAgIHNlbGYuZGVjMSA9IENvbnZCbG9jaygyICogdzEsIHcxKQ0KICAgICAgICBzZWxmLnVwMSA9IG5uLkNvbnZUcmFuc3Bvc2UxZCh3MSwgdzAsIDQsIHN0cmlkZT0yLCBwYWRkaW5nPTEpDQogICAgICAgIHNlbGYuZGVjMCA9IENvbnZCbG9jaygyICogdzAsIHcwKQ0KICAgICAgICBzZWxmLmhlYWQgPSBubi5Db252MWQodzAsIGxlbihIRUFEU1traW5kXSksIDEpDQogICAgICAgIGlmIGtpbmQgIT0gJ1cnOiAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgIyBldmVudCBoZWFkcyBzdGFydCBuZWFyIGEgbG93IHByaW9yIChmb2NhbC1sb3NzIHByYWN0aWNlKQ0KICAgICAgICAgICAgbm4uaW5pdC5jb25zdGFudF8oc2VsZi5oZWFkLmJpYXMsIC00LjApDQoNCiAgICBkZWYgZm9yd2FyZChzZWxmLCB4KToNCiAgICAgICAgbiA9IHguc2hhcGVbLTFdDQogICAgICAgIHBhZCA9ICgtbikgJSA0DQogICAgICAgIGlmIHBhZDoNCiAgICAgICAgICAgIHggPSBGLnBhZCh4LCAoMCwgcGFkKSwgbW9kZT0ncmVmbGVjdCcpDQogICAgICAgIGUwID0gc2VsZi5lbmMwKHgpDQogICAgICAgIGUxID0gc2VsZi5lbmMxKHNlbGYuZG93bjEoZTApKQ0KICAgICAgICBtID0gc2VsZi5taWQoc2VsZi5kb3duMihlMSkpDQogICAgICAgIGQxID0gc2VsZi5kZWMxKHRvcmNoLmNhdChbc2VsZi51cDIobSksIGUxXSwgMSkpDQogICAgICAgIGQwID0gc2VsZi5kZWMwKHRvcmNoLmNhdChbc2VsZi51cDEoZDEpLCBlMF0sIDEpKQ0KICAgICAgICByZXR1cm4gc2VsZi5oZWFkKGQwKVsuLi4sIDpuXQ0KDQoNCmRlZiBuX3BhcmFtcyhtb2RlbCk6DQogICAgcmV0dXJuIHN1bShwLm51bWVsKCkgZm9yIHAgaW4gbW9kZWwucGFyYW1ldGVycygpKQ0K", "robust_beats.py": "...", "decoder.py": "IiIiQmVhdCBkZWNvZGVyIHNoYXJlZCBieSBhcm1zIFQgYW5kIFcgKFBhbmVsIEgsIGRlY2lzaW9uIDIpLCBzbyB0aGF0IFcgYWdhaW5zdCBUIGNvbXBhcmVzIHRyYWNlcywgbm90IGRlY29kZXJzLg0KDQpJbnB1dDogYW4gZXZlbnQgdHJhY2UgKFQ6IHRoZSBSIG9yIFMxIGhlYWQgb3V0cHV0KSBvciBhIHN5bnRoZXRpYyBFQ0cgKFc6IGRlcml2YXRpdmUgZW5lcmd5IG9mIGl0cyA1IHRvIDI1IEh6IGJhbmQpLg0KU3RlcHMsIGFsbCBhdCA1MDAgSHo6DQogIDEuIGRldGVjdGlvbiBlbmVyZ3kgKEVDRykgb3IgdGhlIHRyYWNlIGl0c2VsZiAoZXZlbnRzKTsgNjAgbXMgc21vb3RoaW5nIGZvciBjYW5kaWRhdGVzOw0KICAyLiBleHBlY3RlZCBSUiBmcm9tIGF1dG9jb3JyZWxhdGlvbiAoZnJvemVuIGRldGVjdG9yJ3MgcnJfcHJpb3I6IDMwIHRvIDI0MiBicG0gc2VhcmNoLCBoaWdoZXN0IGxvY2FsIG1heGltdW0sDQogICAgIGhhbGYtbGFnIHN3aXRjaCBhdCAwLjY7IDIwIHMgd2luZG93cyBhY2NlcHRlZCB3aXRoaW4gMC41NSB0byAxLjggeCB0aGUgZ2xvYmFsIHZhbHVlKTsNCiAgMy4gY2FuZGlkYXRlcyBhdCBsZWFzdCAxMjAgbXMgYXBhcnQgd2l0aCBwcm9taW5lbmNlIGF0IGxlYXN0IDIlIG9mIHRoZSBsb2NhbCBiZWF0IGxldmVsOw0KICA0LiBlbWlzc2lvbiBmcm9tIGFtcGxpdHVkZSBhbG9uZSAocmVsYXRpdmUgdG8gdGhlIGxvY2FsIGJlYXQgbGV2ZWwpOw0KICA1LiB0d28tcGFzcyBWaXRlcmJpIG92ZXIgbG9nLVJSIHRyYW5zaXRpb25zIChzaW51cyBTRCAwLjI1LCB0aGVuIDAuMTUgYXJvdW5kIGEgOS1iZWF0IHJ1bm5pbmcgbWVkaWFuOyBwcmVtYXR1cmUNCiAgICAgMC4xMCBvdmVyIDAuNCB0byAxLjAgUlI7IGxvbmcgMC4wNSBvdmVyIDEuMCB0byAxLjggUlI7IG1pc3NlZCBiZWF0IDAuMDEgYXQgMiBhbmQgMyBSUiksIHJlZnJhY3RvcnkNCiAgICAgbWF4KDIwMCBtcywgMC4zNSB4IGV4cGVjdGVkIFJSKSwgcmVzdGFydCBwZW5hbHR5IC0xMiAodGhlIGZyb3plbiBkZXRlY3RvcidzIHZhbHVlcyk7DQogIDYuIGJlYXQgcG9zaXRpb246IGVuZXJneSBjZW50cm9pZCAoRUNHKSBvciB0aGUgdHJhY2UgbWF4aW11bSB3aXRoIHBhcmFib2xpYyByZWZpbmVtZW50IChldmVudHMpOw0KICA3LiBubyBsYWJlbHMgYWNyb3NzIGdhcHMgbG9uZ2VyIHRoYW4gMS44IHggdGhlIGV4cGVjdGVkIFJSLg0KTm90aGluZyBoZXJlIHJlYWRzIGEgZGF0YXNldDsgdGhlIHRyYWNlIGNvbWVzIGZyb20gdGhlIGNhbGxlci4NCiIiIg0KaW1wb3J0IG51bXB5IGFzIG5wDQppbXBvcnQgc2NpcHkuc2lnbmFsIGFzIHNnDQoNCmZyb20gZXh0c3R1ZHkgaW1wb3J0IHJvYnVzdF9iZWF0cyBhcyBSQg0KDQpGUyA9IFJCLkZTDQojIGVtaXNzaW9uIDQgbG9nMihhbXBsaXR1ZGUgLyBsb2NhbCBiZWF0IGxldmVsKSArIDI6IHplcm8gYXQgMC43MSBvZiB0aGUgbG9jYWwgYmVhdCBsZXZlbCAoYXMgaW4gdGhlIGZyb3plbiBkZXRlY3RvciwNCiMgMiBsb2cyICsgMSksIGJ1dCBhIGdvb2QgYmVhdCBzY29yZXMgKzIsIG5vdCArMS4gV2l0aCArMSwgZHJvcHBpbmcgYSBzdHJldGNoIG9mIGJlYXRzIHRocm91Z2ggb25lIHJlc3RhcnQgKC0xMikgYmVhdA0KIyBrZWVwaW5nIGl0IHdoZW5ldmVyIGEgdGhpcmQgb2YgaXRzIGJlYXRzIHdlcmUgbWlzc2luZyAoZWFjaCBtaXNzZWQgYmVhdCBjb3N0cyBhYm91dCAtMy40KTogMTEgdHJ1ZSBiZWF0cyBsb3N0IGluDQojIGEgc2ltdWxhdGVkIHJlY29yZCB3aXRoIGEgMyBzIHBhdXNlDQpQQVJBTVMgPSBkaWN0KGNhbmRfbWluX21zPTEyMCwgY2FuZF9wcm9tPTAuMDIsIHJlZl93aW5fcz01LjAsIHdfYW1wPTQuMCwgYW1wX2NsaXA9KC00LjAsIDEuMCksIGVfYmlhcz0yLjAsDQogICAgICAgICAgICAgIHNpZ21hMT1SQi5QWydzaWdtYTEnXSwgbWF4X2dhcDE9UkIuUFsnbWF4X2dhcDEnXSwgc2lnbWEyPVJCLlBbJ3NpZ21hJ10sIG1heF9nYXAyPVJCLlBbJ21heF9nYXBfcmF0aW8nXSwNCiAgICAgICAgICAgICAgcnJfcmVsX21pbj1SQi5QWydycl9yZWxfbWluJ10sIGdhcF9mbGFnPTEuOCwgY2VudF9tcz02MCkNCg0KDQpkZWYgYmVhdF90cmFjZV9mcm9tX2hlYWRzKHBfYmVhdCwgcF9zMik6DQogICAgIiIiQXJtIFQncyBpbnB1dCB0byB0aGUgZGVjb2RlcjogdGhlIGJlYXQgaGVhZCAoUiBvciBTMSkgbWludXMgdGhlIFMyIGhlYWQsIGZsb29yZWQgYXQgMC4gQW4gUzIgc2VxdWVuY2UgaXMgYXMNCiAgICByZWd1bGFyIGFzIHRoZSBiZWF0IHNlcXVlbmNlOyB3aXRoIFMxIG1pc3NpbmcgaW4gYSBzdHJldGNoLCBhbXBsaXR1ZGUgYW5kIHJoeXRobSBhbG9uZSBwcmVmZXJyZWQgdGhlIFMyIGNoYWluDQogICAgKHNpbXVsYXRlZCB0cmFjZXMpLCBzbyBTMiBldmlkZW5jZSBlbnRlcnMgdGhyb3VnaCB0aGUgdHJhY2UgYW5kIHRoZSBkZWNvZGVyIHN0YXlzIHRoZSBzYW1lIGZvciBUIGFuZCBXLiIiIg0KICAgIHJldHVybiBucC5jbGlwKG5wLmFzYXJyYXkocF9iZWF0LCBucC5mbG9hdDY0KSAtIG5wLmFzYXJyYXkocF9zMiwgbnAuZmxvYXQ2NCksIDAuMCwgTm9uZSkNCg0KDQpkZWYgZWNnX2VuZXJneShlY2c1MDApOg0KICAgICIiIlNxdWFyZWQgNS1wb2ludCBkZXJpdmF0aXZlIG9mIHRoZSA1IHRvIDI1IEh6IGJhbmQgKFBhbi1Ub21wa2lucyBmb3JtKSwgYW5kIHRoZSBiYW5kIGl0c2VsZi4iIiINCiAgICBxID0gUkIuX2ZpbHRfc2VnbWVudHMobnAuYXNhcnJheShlY2c1MDAsIG5wLmZsb2F0NjQpLCBSQi5fc29zKCdxcnMnKSwgWzAsIGxlbihlY2c1MDApXSkNCiAgICBxID0gcSAvIChucC5tZWRpYW4obnAuYWJzKHEpKSArIDFlLTEyKQ0KICAgIGQgPSBucC56ZXJvc19saWtlKHEpDQogICAgZFsyOi0yXSA9ICgtcVs0Ol0gLSAyICogcVszOi0xXSArIDIgKiBxWzE6LTNdICsgcVs6LTRdKSAvIDguMA0KICAgIHJldHVybiBkICoqIDIsIHENCg0KDQpkZWYgX2xvY2FsX2xldmVsKHBrLCBoLCBycmUsIFcpOg0KICAgIHJlZiA9IG5wLmVtcHR5KGxlbihwaykpDQogICAgZm9yIGksIHAgaW4gZW51bWVyYXRlKHBrKToNCiAgICAgICAgYSwgYiA9IG5wLnNlYXJjaHNvcnRlZChwaywgcCAtIFcpLCBucC5zZWFyY2hzb3J0ZWQocGssIHAgKyBXKQ0KICAgICAgICBrID0gbWF4KDIsIGludChyb3VuZCgyICogUEFSQU1TWydyZWZfd2luX3MnXSAvIG1heChycmVbbWluKHAsIGxlbihycmUpIC0gMSldLCAwLjIpKSkpDQogICAgICAgIHJlZltpXSA9IG5wLm1lZGlhbihucC5zb3J0KGhbYTpiXSlbOjotMV1bOmtdKQ0KICAgIHJldHVybiByZWYNCg0KDQpkZWYgX3JlZmluZV9wZWFrKHksIHApOg0KICAgIGlmIDEgPD0gcCA8IGxlbih5KSAtIDE6DQogICAgICAgIHkwLCB5MSwgeTIgPSB5W3AgLSAxXSwgeVtwXSwgeVtwICsgMV0NCiAgICAgICAgZGVuID0geTAgLSAyICogeTEgKyB5Mg0KICAgICAgICByZXR1cm4gcCArIChucC5jbGlwKDAuNSAqICh5MCAtIHkyKSAvIGRlbiwgLTAuNSwgMC41KSBpZiBkZW4gIT0gMCBlbHNlIDAuMCkNCiAgICByZXR1cm4gZmxvYXQocCkNCg0KDQpkZWYgZGVjb2RlX2JlYXRzKHRyYWNlLCBmcywga2luZD0nZXZlbnQnKToNCiAgICAiIiJ0cmFjZTogZXZlbnQgcHJvYmFiaWxpdHkgKGtpbmQ9J2V2ZW50Jykgb3Igc3ludGhldGljIEVDRyAoa2luZD0nZWNnJykgYXQgZnMuDQogICAgUmV0dXJucyBkaWN0KGJlYXRzX3MsIGJlYXRzNTAwLCBtYXNrNTAwIChUcnVlID0gbGFiZWxsZWQpLCBycmVfcywgcmVjKS4iIiINCiAgICB4ID0gUkIucmVzYW1wbGVfdG8obnAubmFuX3RvX251bShucC5hc2FycmF5KHRyYWNlLCBucC5mbG9hdDY0KSksIGZzLCBGUykNCiAgICBuID0gbGVuKHgpDQogICAgaWYga2luZCA9PSAnZWNnJzoNCiAgICAgICAgZSwgcSA9IGVjZ19lbmVyZ3koeCkNCiAgICAgICAgZXJhdyA9IHEgKiogMg0KICAgIGVsaWYga2luZCA9PSAnZXZlbnQnOg0KICAgICAgICBlLCBlcmF3ID0gbnAubWF4aW11bSh4LCAwLjApLCBOb25lDQogICAgZWxzZToNCiAgICAgICAgcmFpc2UgVmFsdWVFcnJvcihraW5kKQ0KICAgIGU2MCA9IFJCLl9tb3ZhdmcoZSwgMC4wNjAgKiBGUykgaWYga2luZCA9PSAnZWNnJyBlbHNlIFJCLl9tb3ZhdmcoZSwgMC4wMjAgKiBGUykNCiAgICBycmUsIGFjZl9zdHJlbmd0aCA9IFJCLnJyX3ByaW9yKGU2MCBpZiBraW5kID09ICdlY2cnIGVsc2UgUkIuX21vdmF2ZyhlLCAwLjA2MCAqIEZTKSkNCiAgICBwaywgcHIgPSBzZy5maW5kX3BlYWtzKGU2MCwgZGlzdGFuY2U9aW50KFBBUkFNU1snY2FuZF9taW5fbXMnXSAvIDEwMDAgKiBGUyksIHByb21pbmVuY2U9MCkNCiAgICBlbXB0eSA9IGRpY3QoYmVhdHNfcz1ucC5hcnJheShbXSksIGJlYXRzNTAwPW5wLmFycmF5KFtdLCBpbnQpLCBtYXNrNTAwPW5wLnplcm9zKG4sIGJvb2wpLCBycmVfcz1ycmUsDQogICAgICAgICAgICAgICAgIHJlYz1kaWN0KG5fYmVhdHM9MCwgYWNmX3N0cmVuZ3RoPWFjZl9zdHJlbmd0aCwgY292ZXJhZ2U9MC4wLCBocl9tZWRpYW49bnAubmFuKSkNCiAgICBpZiBsZW4ocGspIDwgMzoNCiAgICAgICAgcmV0dXJuIGVtcHR5DQogICAgaCA9IGU2MFtwa10NCiAgICByZWYgPSBfbG9jYWxfbGV2ZWwocGssIGgsIHJyZSwgaW50KFBBUkFNU1sncmVmX3dpbl9zJ10gKiBGUykpDQogICAga2VlcCA9IHByWydwcm9taW5lbmNlcyddID49IFBBUkFNU1snY2FuZF9wcm9tJ10gKiByZWYNCiAgICBwaywgaCwgcmVmID0gcGtba2VlcF0sIGhba2VlcF0sIHJlZltrZWVwXQ0KICAgIGlmIGxlbihwaykgPCAzOg0KICAgICAgICByZXR1cm4gZW1wdHkNCiAgICBlbSA9IFBBUkFNU1snd19hbXAnXSAqIG5wLmNsaXAobnAubG9nMihoIC8gKHJlZiArIDFlLTEyKSArIDFlLTEyKSwgKlBBUkFNU1snYW1wX2NsaXAnXSkgKyBQQVJBTVNbJ2VfYmlhcyddDQogICAgaWYga2luZCA9PSAnZWNnJzoNCiAgICAgICAgaGFsZiA9IGludChQQVJBTVNbJ2NlbnRfbXMnXSAvIDEwMDAgKiBGUykNCiAgICAgICAgcG9zID0gbnAuYXJyYXkoW1JCLl9jZW50cm9pZChlcmF3LCBwLCBoYWxmLCAyKSBmb3IgcCBpbiBwa10pDQogICAgZWxzZToNCiAgICAgICAgcG9zID0gbnAuYXJyYXkoW19yZWZpbmVfcGVhayhlNjAsIHApIGZvciBwIGluIHBrXSkNCiAgICB0YyA9IG5wLnJvdW5kKHBvcykuYXN0eXBlKGludCkNCiAgICBvcmRlciA9IG5wLmFyZ3NvcnQodGMpDQogICAgdGMsIGVtX28gPSB0Y1tvcmRlcl0sIGVtW29yZGVyXQ0KICAgIHBhdGggPSBSQi5fdml0ZXJiaSh0YywgZW1fbywgcnJlW25wLmNsaXAodGMsIDAsIG4gLSAxKV0sIEZTLCBbXSwgbiwgUEFSQU1TWydzaWdtYTEnXSwgUEFSQU1TWydtYXhfZ2FwMSddKQ0KICAgIGJlYXRzID0gdGNbcGF0aF0NCiAgICBpZiBsZW4oYmVhdHMpID49IDU6DQogICAgICAgIHJyID0gbnAuZGlmZihiZWF0cykgLyBGUw0KICAgICAgICBva3JyID0gcnIgPCAxLjcgKiBucC5tZWRpYW4ocnIpDQogICAgICAgIGlmIG9rcnIuc3VtKCkgPj0gMzoNCiAgICAgICAgICAgIG1pZCA9IDAuNSAqIChiZWF0c1sxOl0gKyBiZWF0c1s6LTFdKQ0KICAgICAgICAgICAgayA9IG1pbig5LCBpbnQob2tyci5zdW0oKSkgfCAxKQ0KICAgICAgICAgICAgcnJfcnVuID0gc2cubWVkZmlsdChycltva3JyXSwgaykgaWYgb2tyci5zdW0oKSA+PSBrIGVsc2UgbnAuZnVsbChva3JyLnN1bSgpLCBucC5tZWRpYW4ocnJbb2tycl0pKQ0KICAgICAgICAgICAgcnJlID0gbnAuaW50ZXJwKG5wLmFyYW5nZShuKSwgbWlkW29rcnJdLCBycl9ydW4pDQogICAgICAgICAgICBwYXRoID0gUkIuX3ZpdGVyYmkodGMsIGVtX28sIHJyZVtucC5jbGlwKHRjLCAwLCBuIC0gMSldLCBGUywgW10sIG4sIFBBUkFNU1snc2lnbWEyJ10sIFBBUkFNU1snbWF4X2dhcDInXSwNCiAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICBQQVJBTVNbJ3JyX3JlbF9taW4nXSkNCiAgICBzZWwgPSBvcmRlcltwYXRoXQ0KICAgIGlmIGtpbmQgPT0gJ2VjZyc6DQogICAgICAgIGIgPSBucC5hcnJheShbUkIuX2NlbnRyb2lkKGVyYXcsIHBrW2ldLCBpbnQobWluKFBBUkFNU1snY2VudF9tcyddIC8gMTAwMCwgMC4xNSAqIHJyZVttaW4ocGtbaV0sIG4gLSAxKV0pICogRlMpLCAyKQ0KICAgICAgICAgICAgICAgICAgICAgIGZvciBpIGluIHNlbF0pDQogICAgZWxzZToNCiAgICAgICAgYiA9IHBvc1tzZWxdDQogICAgYjUwMCA9IG5wLnJvdW5kKGIpLmFzdHlwZShpbnQpDQogICAgbWFzayA9IG5wLm9uZXMobiwgYm9vbCkNCiAgICBycmVfYiA9IHJyZVtucC5jbGlwKGI1MDAsIDAsIG4gLSAxKV0gKiBGUw0KICAgIGZvciBhLCBjLCByIGluIHppcChiNTAwWzotMV0sIGI1MDBbMTpdLCBycmVfYls6LTFdKToNCiAgICAgICAgaWYgYyAtIGEgPiBQQVJBTVNbJ2dhcF9mbGFnJ10gKiByOg0KICAgICAgICAgICAgbWFza1thOmNdID0gRmFsc2UNCiAgICBpZiBsZW4oYjUwMCk6DQogICAgICAgIGlmIGI1MDBbMF0gPiBQQVJBTVNbJ2dhcF9mbGFnJ10gKiBycmVfYlswXToNCiAgICAgICAgICAgIG1hc2tbOmI1MDBbMF1dID0gRmFsc2UNCiAgICAgICAgaWYgbiAtIGI1MDBbLTFdID4gUEFSQU1TWydnYXBfZmxhZyddICogcnJlX2JbLTFdOg0KICAgICAgICAgICAgbWFza1tiNTAwWy0xXTpdID0gRmFsc2UNCiAgICByZWMgPSBkaWN0KG5fYmVhdHM9aW50KGxlbihiNTAwKSksIGFjZl9zdHJlbmd0aD1mbG9hdChhY2Zfc3RyZW5ndGgpLCBjb3ZlcmFnZT1mbG9hdChtYXNrLm1lYW4oKSksDQogICAgICAgICAgICAgICBocl9tZWRpYW49ZmxvYXQoNjAgKiBGUyAvIG5wLm1lZGlhbihucC5kaWZmKGI1MDApKSkgaWYgbGVuKGI1MDApID4gMiBlbHNlIG5wLm5hbiwNCiAgICAgICAgICAgICAgIG5fY2FuZGlkYXRlcz1pbnQobGVuKHBrKSkpDQogICAgcmV0dXJuIGRpY3QoYmVhdHNfcz1iIC8gRlMsIGJlYXRzNTAwPWI1MDAsIG1hc2s1MDA9bWFzaywgcnJlX3M9cnJlLCByZWM9cmVjKQ0K", "corpus.py": "IiIiVHJhaW5pbmcgcGFjayBmb3IgQ29sYWIsIGJ1aWx0IGxvY2FsbHkgZnJvbSB0aGUgc2l4IHBhaXJlZCBQQ0ctRUNHIGRhdGFzZXRzIG9ubHkuDQoNCkZpbGVzIGluIDxvdXQ+Lzx2ZXJzaW9uPi86DQogIHBjZ19icF8xa19pbnQxNi5ucHkgICBldmVyeSB1c2FibGUgcmVjb3JkLCBiYW5kLXBhc3NlZCAyNSB0byA0MDAgSHogYXQgMSBrSHogKGZyb250ZW5kLmJhbmRwYXNzKSwgY29uY2F0ZW5hdGVkOw0KICAgICAgICAgICAgICAgICAgICAgICAgcmVjb3JkIGsgb2NjdXBpZXMgW29mZnNldF9rLCBvZmZzZXRfayArIG5faykgYW5kIGVxdWFscyBpbnQxNiAvIHNjYWxlX2sNCiAgYmluX21lZGlhbnMubnB5ICAgICAgIChyZWNvcmRzLCAyNCkgcGVyLWJpbiBsb2ctcG93ZXIgbWVkaWFucyBvZiBlYWNoIHdob2xlIHJlY29yZGluZyAoZnJvbnRlbmQuYmluX21lZGlhbnMpDQogIHJlY29yZHMuY3N2ICAgICAgICAgICByZWNvcmRfaWQsIGRhdGFzZXQsIHJlY29yZCwgc3ViamVjdCwgZ3JvdXAsIHJvbGUsIG9mZnNldCwgbl9zYW1wbGVzLCBzY2FsZSwgYW5kIGNvdW50cw0KICBldmVudHMubnB6ICAgICAgICAgICAgcGVyIGJlYXQsIGNvbmNhdGVuYXRlZCBpbiByZWNvcmQgb3JkZXI6IHJlY29yZF9pZCwgcl9zLCByX25leHRfcywgczFfcywgczJfcywgcl90YXJnZXRfcywNCiAgICAgICAgICAgICAgICAgICAgICAgIHMxX29uX3MsIHMxX29mZl9zLCBzMl9vbl9zLCBzMl9vZmZfcywgaHIsIHJyX21zLCB2YWxpZA0KICBtYW5pZmVzdC5qc29uICAgICAgICAgU0hBLTI1NiBvZiBldmVyeSBmaWxlOyB0aGUgZGV0ZWN0b3IsIGxhYmVsLWxhdywgYW5kIHNwbGl0LXRhYmxlIGhhc2hlczsgY291bnRzIHBlciBkYXRhc2V0DQogICAgICAgICAgICAgICAgICAgICAgICBhbmQgcm9sZQ0KTG9zcyBtYXNrcyBmb2xsb3cgZnJvbSBldmVudHM6IHRoZSBjeWNsZXMgW3Jfcywgcl9uZXh0X3MpIG9mIHZhbGlkIGJlYXRzOyBldmVyeXRoaW5nIGVsc2UgY2FycmllcyBubyBsb3NzLg0KVXNhZ2U6ICBweXRob24gLW0gZXh0c3R1ZHkuY29ycHVzIFstLXZlcnNpb24gcGFpcmVkX3YyXSBbLS1lY2ddICAgKGRlZmF1bHQ6IGNvbmZpZy5QQUNLLCBsYWJlbHMgZnJvbSBjb25maWcuTEFCRUxTKQ0KIiIiDQppbXBvcnQgYXJncGFyc2UsIGhhc2hsaWIsIGpzb24sIG9zLCB0aW1lDQpmcm9tIGRhdGV0aW1lIGltcG9ydCBkYXRldGltZSwgdGltZXpvbmUNCmltcG9ydCBudW1weSBhcyBucA0KaW1wb3J0IHBhbmRhcyBhcyBwZA0KDQpmcm9tIGV4dHN0dWR5IGltcG9ydCBmcm9udGVuZCBhcyBGDQpmcm9tIGV4dHN0dWR5LmNvbmZpZyBpbXBvcnQgTE9DQUwsIExBQkVMUywgUEFDSw0KDQpFVkVOVF9LRVlTID0gKCdyX3MnLCAncl9uZXh0X3MnLCAnczFfcycsICdzMl9zJywgJ3JfdGFyZ2V0X3MnLCAnczFfb25fcycsICdzMV9vZmZfcycsICdzMl9vbl9zJywgJ3MyX29mZl9zJywgJ2hyJywgJ3JyX21zJywgJ3ZhbGlkJykNCklOVDE2X1RBUkdFVCA9IDMwMDAwLjANCg0KDQpkZWYgc2hhMjU2KHBhdGgsIGNodW5rPTEgPDwgMjIpOg0KICAgIGggPSBoYXNobGliLnNoYTI1NigpDQogICAgd2l0aCBvcGVuKHBhdGgsICdyYicpIGFzIGY6DQogICAgICAgIGZvciBiIGluIGl0ZXIobGFtYmRhOiBmLnJlYWQoY2h1bmspLCBiJycpOg0KICAgICAgICAgICAgaC51cGRhdGUoYikNCiAgICByZXR1cm4gaC5oZXhkaWdlc3QoKQ0KDQoNCmRlZiBfb25lKHJvdyk6DQogICAgeiA9IG5wLmxvYWQob3MucGF0aC5qb2luKExPQ0FMWydoYXJtb25pc2VkJ10sIHJvd1snZGF0YXNldCddLCByb3dbJ3JlY29yZCddICsgJy5ucHonKSkNCiAgICB4ID0gRi5iYW5kcGFzcyhGLnRvXzFrKHpbJ3BjZyddLCBmbG9hdCh6Wydmc19wY2cnXSkpKQ0KICAgIG1lZCA9IEYuYmluX21lZGlhbnMoeCkNCiAgICBhID0gbnAucGVyY2VudGlsZShucC5hYnMoeCksIDk5Ljk5KSArIDFlLTEyDQogICAgc2NhbGUgPSBJTlQxNl9UQVJHRVQgLyBhDQogICAgcSA9IG5wLmNsaXAobnAucm91bmQoeCAqIHNjYWxlKSwgLTMyNzY3LCAzMjc2NykuYXN0eXBlKG5wLmludDE2KQ0KICAgIEwgPSBucC5sb2FkKG9zLnBhdGguam9pbihMT0NBTFsncGhkJ10sICdkZXJpdmVkJywgTEFCRUxTLCByb3dbJ2RhdGFzZXQnXSwgcm93WydyZWNvcmQnXSArICcubnB6JykpDQogICAgZXYgPSB7azogbnAuYXNhcnJheShMW2tdKSBmb3IgayBpbiBFVkVOVF9LRVlTfQ0KICAgIHJldHVybiByb3csIHEsIGZsb2F0KHNjYWxlKSwgbWVkLmFzdHlwZShucC5mbG9hdDMyKSwgZXYsIHN0cihMWydsYXdzX3NoYTI1NiddKQ0KDQoNCmRlZiBidWlsZCh2ZXJzaW9uPVBBQ0ssIHdvcmtlcnM9MTIpOg0KICAgIGZyb20gbXVsdGlwcm9jZXNzaW5nIGltcG9ydCBQb29sDQogICAgZnJvbSBleHRzdHVkeSBpbXBvcnQgc3BsaXRzLCBxcnMNCiAgICBvdXQgPSBvcy5wYXRoLmpvaW4oTE9DQUxbJ3BoZCddLCAnZGVyaXZlZCcsICdjb3JwdXMnLCB2ZXJzaW9uKQ0KICAgIG9zLm1ha2VkaXJzKG91dCwgZXhpc3Rfb2s9VHJ1ZSkNCiAgICBsYWIgPSBwZC5yZWFkX2Nzdihvcy5wYXRoLmpvaW4oTE9DQUxbJ3BoZCddLCAnZGVyaXZlZCcsIExBQkVMUywgJ2xhYmVsc19yZWNvcmRzLmNzdicpKQ0KICAgIFQgPSBzcGxpdHMucGFpcmVkX3NwbGl0cygpDQogICAgVCA9IFQubWVyZ2UobGFiW1snZGF0YXNldCcsICdyZWNvcmQnLCAnbl9iZWF0cycsICduX3ZhbGlkJywgJ3ZhbGlkX3MnXV0sIG9uPVsnZGF0YXNldCcsICdyZWNvcmQnXSwgaG93PSdsZWZ0JykNCiAgICBUID0gVFtUWyduX3ZhbGlkJ10uZmlsbG5hKDApID4gMF0uc29ydF92YWx1ZXMoWydkYXRhc2V0JywgJ3JlY29yZCddKS5yZXNldF9pbmRleChkcm9wPVRydWUpDQogICAgcm93cyA9IFQudG9fZGljdCgncmVjb3JkcycpDQogICAgdDAgPSB0aW1lLnRpbWUoKQ0KICAgIHdpdGggUG9vbCh3b3JrZXJzKSBhcyBwb29sOg0KICAgICAgICByZXMgPSBwb29sLm1hcChfb25lLCByb3dzLCBjaHVua3NpemU9NCkNCiAgICBuX3RvdCA9IHN1bShsZW4ocSkgZm9yIF8sIHEsIF8sIF8sIF8sIF8gaW4gcmVzKQ0KICAgIHBjZ19wYXRoID0gb3MucGF0aC5qb2luKG91dCwgJ3BjZ19icF8xa19pbnQxNi5ucHknKQ0KICAgIGJpZyA9IG5wLmxpYi5mb3JtYXQub3Blbl9tZW1tYXAocGNnX3BhdGgsIG1vZGU9J3crJywgZHR5cGU9bnAuaW50MTYsIHNoYXBlPShuX3RvdCwpKQ0KICAgIHJlY3MsIG1lZHMsIGV2cywgbGF3cyA9IFtdLCBbXSwge2s6IFtdIGZvciBrIGluIEVWRU5UX0tFWVMgKyAoJ3JlY29yZF9pZCcsKX0sIHNldCgpDQogICAgb2ZmID0gMA0KICAgIGZvciBrLCAocm93LCBxLCBzY2FsZSwgbWVkLCBldiwgbGgpIGluIGVudW1lcmF0ZShyZXMpOg0KICAgICAgICBhc3NlcnQgKHJvd1snZGF0YXNldCddLCByb3dbJ3JlY29yZCddKSA9PSAocm93c1trXVsnZGF0YXNldCddLCByb3dzW2tdWydyZWNvcmQnXSkgICAgICAjIElEcyB0cmF2ZWwgd2l0aCBkYXRhDQogICAgICAgIGJpZ1tvZmY6b2ZmICsgbGVuKHEpXSA9IHENCiAgICAgICAgcmVjcy5hcHBlbmQoZGljdChyZWNvcmRfaWQ9aywgZGF0YXNldD1yb3dbJ2RhdGFzZXQnXSwgcmVjb3JkPXJvd1sncmVjb3JkJ10sIHN1YmplY3Q9cm93WydzdWJqZWN0J10sIGdyb3VwPXJvd1snZ3JvdXAnXSwNCiAgICAgICAgICAgICAgICAgICAgICAgICByb2xlPXJvd1sncm9sZSddLCBvZmZzZXQ9b2ZmLCBuX3NhbXBsZXM9bGVuKHEpLCBzY2FsZT1zY2FsZSwgZHVyYXRpb25fcz1sZW4ocSkgLyBGLkZTLA0KICAgICAgICAgICAgICAgICAgICAgICAgIG5fYmVhdHM9aW50KHJvd1snbl9iZWF0cyddKSwgbl92YWxpZD1pbnQocm93WyduX3ZhbGlkJ10pLCB2YWxpZF9zPWZsb2F0KHJvd1sndmFsaWRfcyddKSkpDQogICAgICAgIG1lZHMuYXBwZW5kKG1lZCkNCiAgICAgICAgZm9yIGtleSBpbiBFVkVOVF9LRVlTOg0KICAgICAgICAgICAgZXZzW2tleV0uYXBwZW5kKGV2W2tleV0pDQogICAgICAgIGV2c1sncmVjb3JkX2lkJ10uYXBwZW5kKG5wLmZ1bGwobGVuKGV2WydyX3MnXSksIGssIG5wLmludDMyKSkNCiAgICAgICAgbGF3cy5hZGQobGgpDQogICAgICAgIG9mZiArPSBsZW4ocSkNCiAgICBiaWcuZmx1c2goKTsgZGVsIGJpZw0KICAgIGFzc2VydCBsZW4obGF3cykgPT0gMSwgJ2xhYmVscyBmcm9tIG1vcmUgdGhhbiBvbmUgbGF3IHNldCcNCiAgICBSID0gcGQuRGF0YUZyYW1lKHJlY3MpDQogICAgUi50b19jc3Yob3MucGF0aC5qb2luKG91dCwgJ3JlY29yZHMuY3N2JyksIGluZGV4PUZhbHNlKQ0KICAgIG5wLnNhdmUob3MucGF0aC5qb2luKG91dCwgJ2Jpbl9tZWRpYW5zLm5weScpLCBucC5zdGFjayhtZWRzKSkNCiAgICBucC5zYXZlel9jb21wcmVzc2VkKG9zLnBhdGguam9pbihvdXQsICdldmVudHMubnB6JyksICoqe2s6IG5wLmNvbmNhdGVuYXRlKHYpIGZvciBrLCB2IGluIGV2cy5pdGVtcygpfSkNCiAgICBjb3VudHMgPSBSLmdyb3VwYnkoWydkYXRhc2V0JywgJ3JvbGUnXSkuYWdnKHJlY29yZHM9KCdyZWNvcmQnLCAnc2l6ZScpLCBzdWJqZWN0cz0oJ2dyb3VwJywgJ251bmlxdWUnKSwNCiAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgIHZhbGlkX2JlYXRzPSgnbl92YWxpZCcsICdzdW0nKSwgdmFsaWRfaG91cnM9KCd2YWxpZF9zJywgbGFtYmRhIHM6IHJvdW5kKHMuc3VtKCkgLyAzNjAwLCAzKSkpDQogICAgbWFuID0gZGljdCh2ZXJzaW9uPXZlcnNpb24sIGxhYmVscz1MQUJFTFMsIGNyZWF0ZWRfdXRjPWRhdGV0aW1lLm5vdyh0aW1lem9uZS51dGMpLmlzb2Zvcm1hdCh0aW1lc3BlYz0nc2Vjb25kcycpLA0KICAgICAgICAgICAgICAgZnM9Ri5GUywgYmFuZF9oej1GLkJBTkQsIGZyYW1lcz1kaWN0KHdpbj1GLldJTiwgaG9wPUYuSE9QLCBuZmZ0PUYuTkZGVCwgYmlucz1bRi5CSU5TLnN0YXJ0LCBGLkJJTlMuc3RvcF0pLA0KICAgICAgICAgICAgICAgaW50MTZfdGFyZ2V0PUlOVDE2X1RBUkdFVCwgbl9yZWNvcmRzPWxlbihSKSwgbl9zYW1wbGVzPWludChuX3RvdCksDQogICAgICAgICAgICAgICBkZXRlY3Rvcl9zaGEyNTY9cXJzLkZST1pFTl9TSEEyNTYsIGxhd3Nfc2hhMjU2PWxhd3MucG9wKCksIHNwbGl0X3RhYmxlX3NoYTI1Nj1zcGxpdHMudGFibGVfaGFzaChUWw0KICAgICAgICAgICAgICAgICAgIFsnZGF0YXNldCcsICdzdWJqZWN0JywgJ2dyb3VwJywgJ3JlY29yZCcsICdzaXRlJywgJ3JvbGUnLCAnZHVyYXRpb25fcycsICduX3JwZWFrcycsICdocl9tZWRpYW4nXV0pLA0KICAgICAgICAgICAgICAgZmlsZXM9e2Y6IHNoYTI1Nihvcy5wYXRoLmpvaW4ob3V0LCBmKSkgZm9yIGYgaW4gKCdwY2dfYnBfMWtfaW50MTYubnB5JywgJ2Jpbl9tZWRpYW5zLm5weScsICdyZWNvcmRzLmNzdicsICdldmVudHMubnB6Jyl9LA0KICAgICAgICAgICAgICAgY291bnRzPXsnJXMvJXMnICUgazogdiBmb3IgaywgdiBpbiBjb3VudHMudG9fZGljdCgnaW5kZXgnKS5pdGVtcygpfSkNCiAgICBqc29uLmR1bXAobWFuLCBvcGVuKG9zLnBhdGguam9pbihvdXQsICdtYW5pZmVzdC5qc29uJyksICd3JyksIGluZGVudD0xLCBkZWZhdWx0PXN0cikNCiAgICBwcmludCgncGFjayAlczogJWQgcmVjb3JkcywgJS4xZiBoLCAlLjBmIE1CLCAlLjBmIHMnICUgKHZlcnNpb24sIGxlbihSKSwgbl90b3QgLyBGLkZTIC8gMzYwMCwNCiAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgb3MucGF0aC5nZXRzaXplKHBjZ19wYXRoKSAvIDFlNiwgdGltZS50aW1lKCkgLSB0MCkpDQogICAgcHJpbnQoY291bnRzLnRvX3N0cmluZygpKQ0KICAgIHJldHVybiBtYW4NCg0KDQpkZWYgbG9hZChmb2xkZXIpOg0KICAgICIiIkNvbGFiIHNpZGU6IG1lbW9yeS1tYXBwZWQgUENHLCByZWNvcmQgdGFibGUsIG1lZGlhbnMsIGV2ZW50czsgY2hlY2tzIGV2ZXJ5IGhhc2ggaW4gdGhlIG1hbmlmZXN0LiIiIg0KICAgIG1hbiA9IGpzb24ubG9hZChvcGVuKG9zLnBhdGguam9pbihmb2xkZXIsICdtYW5pZmVzdC5qc29uJykpKQ0KICAgIGZvciBmLCBoIGluIG1hblsnZmlsZXMnXS5pdGVtcygpOg0KICAgICAgICBhc3NlcnQgc2hhMjU2KG9zLnBhdGguam9pbihmb2xkZXIsIGYpKSA9PSBoLCAnaGFzaCBtaXNtYXRjaDogJXMnICUgZg0KICAgIHBjZyA9IG5wLmxvYWQob3MucGF0aC5qb2luKGZvbGRlciwgJ3BjZ19icF8xa19pbnQxNi5ucHknKSwgbW1hcF9tb2RlPSdyJykNCiAgICBSID0gcGQucmVhZF9jc3Yob3MucGF0aC5qb2luKGZvbGRlciwgJ3JlY29yZHMuY3N2JykpDQogICAgbWVkID0gbnAubG9hZChvcy5wYXRoLmpvaW4oZm9sZGVyLCAnYmluX21lZGlhbnMubnB5JykpDQogICAgZXYgPSBkaWN0KG5wLmxvYWQob3MucGF0aC5qb2luKGZvbGRlciwgJ2V2ZW50cy5ucHonKSkpDQogICAgYXNzZXJ0IGxlbihSKSA9PSBsZW4obWVkKSA9PSBtYW5bJ25fcmVjb3JkcyddIGFuZCBsZW4ocGNnKSA9PSBtYW5bJ25fc2FtcGxlcyddDQogICAgcmV0dXJuIG1hbiwgcGNnLCBSLCBtZWQsIGV2DQoNCg0KZGVmIHJlY29yZF9zaWduYWwocGNnLCBSLCBrKToNCiAgICByID0gUi5pbG9jW2tdDQogICAgcmV0dXJuIHBjZ1tpbnQoci5vZmZzZXQpOmludChyLm9mZnNldCkgKyBpbnQoci5uX3NhbXBsZXMpXS5hc3R5cGUobnAuZmxvYXQzMikgLyBucC5mbG9hdDMyKHIuc2NhbGUpDQoNCg0KIyAtLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0gRUNHIHRhcmdldHMgZm9yIFcgKGFkZGVkIDE3IE9jdCAyMDI2KQ0KRUNHX0ZTID0gMjUwDQpFQ0dfSU5UMTZfUEVSX1VOSVQgPSA4MDAwLjAgICAgICAgICAgICAgICMgb25lIHVuaXQgPSB0aGUgcmVjb3JkJ3MgbWVkaWFuIFIgYW1wbGl0dWRlOyBjbGlwcGVkIGF0IGFib3V0ICsvLSA0IHVuaXRzDQoNCg0KZGVmIF9lY2dfb25lKHJvdyk6DQogICAgIiIiVGhlIHRlYWNoZXIgbGVhZCBvZiBvbmUgcmVjb3JkOiAwLjUgdG8gNDAgSHogKFNPUywgcG9sZSBjaGVjayksIFFSUyBtYWRlIHBvc2l0aXZlLCBzY2FsZWQgdG8gdGhlIG1lZGlhbiBSIGFtcGxpdHVkZQ0KICAgIGF0IHRoZSB0ZWFjaGVyIGJlYXRzLCByZXNhbXBsZWQgdG8gMjUwIEh6LCBsZW5ndGggPSBQQ0cgc2FtcGxlcyBhdCAxIGtIeiAvLyA0IChzYW1wbGUgayBhdCBrIC8gMjUwIHMpLiIiIg0KICAgIGltcG9ydCBqc29uIGFzIF9qc29uLCBzeXMgYXMgX3N5cw0KICAgIGltcG9ydCBzY2lweS5zaWduYWwgYXMgc2cNCiAgICBpZiBMT0NBTFsncGNnZWNnX2NvZGUnXSBub3QgaW4gX3N5cy5wYXRoOg0KICAgICAgICBfc3lzLnBhdGguaW5zZXJ0KDAsIExPQ0FMWydwY2dlY2dfY29kZSddKQ0KICAgIGZyb20gcGNnZWNnIGltcG9ydCByZWFkZXJzIGFzIFJkLCBzaWduYWxzIGFzIFNnDQogICAgZnJvbSBleHRzdHVkeS5jb25maWcgaW1wb3J0IExFQURfT1JERVINCiAgICBiID0gbnAubG9hZChvcy5wYXRoLmpvaW4oTE9DQUxbJ2JlYXRzJ10sIHJvd1snZGF0YXNldCddLCByb3dbJ3JlY29yZCddICsgJy5ucHonKSkNCiAgICBsZWFkID0gc3RyKGJbJ2xlYWQnXSkNCiAgICBpZiByb3dbJ2RhdGFzZXQnXSBpbiBMRUFEX09SREVSOg0KICAgICAgICByZWMgPSBbciBmb3IgciBpbiBSZC5sb2FkKHR1cGxlKF9qc29uLmxvYWRzKHJvd1sna2V5J10pKSkgaWYgci5yZWNvcmQgPT0gcm93WydyZWNvcmQnXV1bMF0NCiAgICAgICAgcG9vbCA9IHtyZWMuZWNnX2xlYWQ6IHJlYy5lY2csICoqcmVjLmVjZ19hbHR9DQogICAgICAgIGU1MDAgPSBTZy5yZXNhbXBsZShwb29sW2xlYWRdLCByZWMuZnNfZWNnLCA1MDApDQogICAgZWxzZToNCiAgICAgICAgZTUwMCA9IG5wLmFzYXJyYXkobnAubG9hZChvcy5wYXRoLmpvaW4oTE9DQUxbJ2hhcm1vbmlzZWQnXSwgcm93WydkYXRhc2V0J10sIHJvd1sncmVjb3JkJ10gKyAnLm5weicpKVsnZWNnJ10sIG5wLmZsb2F0NjQpDQogICAgZTUwMCA9IG5wLm5hbl90b19udW0obnAuYXNhcnJheShlNTAwLCBucC5mbG9hdDY0KSkNCiAgICBzb3MgPSBzZy5idXR0ZXIoMiwgWzAuNSwgNDAuMF0sIGJ0eXBlPSdiYW5kcGFzcycsIGZzPTUwMCwgb3V0cHV0PSdzb3MnKQ0KICAgIGFzc2VydCBucC5hYnMoc2cuc29zMnpwayhzb3MpWzFdKS5tYXgoKSA8IDAuOTk5OQ0KICAgIHggPSBzZy5zb3NmaWx0ZmlsdChzb3MsIGU1MDApICogZmxvYXQocm93Wydwb2xhcml0eSddIG9yIDEuMCkNCiAgICBiZWF0cyA9IG5wLmFzYXJyYXkoYlsnYmVhdHMnXSwgaW50KQ0KICAgIHBlYWtzID0gW25wLm1heCh4W21heCgwLCBwIC0gMjApOnAgKyAyMV0pIGZvciBwIGluIGJlYXRzIGlmIDAgPD0gcCA8IGxlbih4KV0NCiAgICBhbXAgPSBmbG9hdChucC5tZWRpYW4ocGVha3MpKSBpZiBsZW4ocGVha3MpID49IDMgYW5kIG5wLm1lZGlhbihwZWFrcykgPiAwIGVsc2UgZmxvYXQobnAucGVyY2VudGlsZShucC5hYnMoeCksIDk5KSkNCiAgICB5ID0gc2cucmVzYW1wbGVfcG9seSh4IC8gKGFtcCArIDFlLTEyKSwgMSwgMikNCiAgICBuID0gaW50KHJvd1snbl9zYW1wbGVzJ10pIC8vIDQNCiAgICBvdXQgPSBucC56ZXJvcyhuLCBucC5mbG9hdDMyKQ0KICAgIG91dFs6bWluKG4sIGxlbih5KSldID0geVs6bl0NCiAgICBxID0gbnAuY2xpcChucC5yb3VuZChvdXQgKiBFQ0dfSU5UMTZfUEVSX1VOSVQpLCAtMzI3NjcsIDMyNzY3KS5hc3R5cGUobnAuaW50MTYpDQogICAgcmV0dXJuIHJvd1sncmVjb3JkX2lkJ10sIHEsIGRpY3QobGVhZD1sZWFkLCBwb2xhcml0eT1mbG9hdChyb3dbJ3BvbGFyaXR5J10gb3IgMS4wKSwgcl9hbXBsaXR1ZGU9YW1wLA0KICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgIG5fc2hvcnQ9aW50KG1heCgwLCBuIC0gbGVuKHkpKSkpDQoNCg0KZGVmIGJ1aWxkX2VjZyh2ZXJzaW9uPVBBQ0ssIHdvcmtlcnM9MTIpOg0KICAgICIiIkFkZHMgZWNnXzI1MF9pbnQxNi5ucHksIGVjZ19yZWNvcmRzLmNzdiBhbmQgZWNnX21hbmlmZXN0Lmpzb24gYmVzaWRlIHRoZSBwYWNrOyB0aGUgcGFjaydzIG93biBmaWxlcyBhbmQgaXRzDQogICAgbWFuaWZlc3Qgc3RheSB1bnRvdWNoZWQgKHRoZWlyIGhhc2hlcyBhcmUgZnJvemVuIGluIHRoZSBwcmUtcmVnaXN0cmF0aW9uKS4iIiINCiAgICBmcm9tIG11bHRpcHJvY2Vzc2luZyBpbXBvcnQgUG9vbA0KICAgIG91dCA9IG9zLnBhdGguam9pbihMT0NBTFsncGhkJ10sICdkZXJpdmVkJywgJ2NvcnB1cycsIHZlcnNpb24pDQogICAgUiA9IHBkLnJlYWRfY3N2KG9zLnBhdGguam9pbihvdXQsICdyZWNvcmRzLmNzdicpKQ0KICAgIEEgPSBwZC5yZWFkX2Nzdihvcy5wYXRoLmpvaW4oTE9DQUxbJ2JlYXRzJ10sICdhdWRpdF92Mi5jc3YnKSwgbG93X21lbW9yeT1GYWxzZSlbWydkYXRhc2V0JywgJ3JlY29yZCcsICdwb2xhcml0eSddXQ0KICAgIEsgPSBwZC5yZWFkX2NzdihMT0NBTFsnYXVkaXRfdGFibGUnXSwgbG93X21lbW9yeT1GYWxzZSlbWydkYXRhc2V0JywgJ3JlY29yZCcsICdrZXknXV0NCiAgICBSMiA9IFIubWVyZ2UoQSwgb249WydkYXRhc2V0JywgJ3JlY29yZCddLCBob3c9J2xlZnQnKS5tZXJnZShLLCBvbj1bJ2RhdGFzZXQnLCAncmVjb3JkJ10sIGhvdz0nbGVmdCcpDQogICAgYXNzZXJ0IGxlbihSMikgPT0gbGVuKFIpIGFuZCBSMlsncmVjb3JkX2lkJ10udG9saXN0KCkgPT0gUlsncmVjb3JkX2lkJ10udG9saXN0KCkNCiAgICB0MCA9IHRpbWUudGltZSgpDQogICAgd2l0aCBQb29sKHdvcmtlcnMpIGFzIHBvb2w6DQogICAgICAgIHJlcyA9IHBvb2wubWFwKF9lY2dfb25lLCBSMi50b19kaWN0KCdyZWNvcmRzJyksIGNodW5rc2l6ZT00KQ0KICAgIHJlcy5zb3J0KGtleT1sYW1iZGEgcjogclswXSkNCiAgICBhc3NlcnQgW3JbMF0gZm9yIHIgaW4gcmVzXSA9PSBSWydyZWNvcmRfaWQnXS50b2xpc3QoKQ0KICAgIGJpZyA9IG5wLmNvbmNhdGVuYXRlKFtxIGZvciBfLCBxLCBfIGluIHJlc10pDQogICAgcm93cywgb2ZmID0gW10sIDANCiAgICBmb3IgKGssIHEsIGluZm8pLCAoXywgcikgaW4gemlwKHJlcywgUi5pdGVycm93cygpKToNCiAgICAgICAgYXNzZXJ0IGxlbihxKSA9PSBpbnQoci5uX3NhbXBsZXMpIC8vIDQNCiAgICAgICAgcm93cy5hcHBlbmQoZGljdChyZWNvcmRfaWQ9aywgZGF0YXNldD1yLmRhdGFzZXQsIHJlY29yZD1yLnJlY29yZCwgZWNnX29mZnNldD1vZmYsIGVjZ19uPWxlbihxKSwgKippbmZvKSkNCiAgICAgICAgb2ZmICs9IGxlbihxKQ0KICAgIG5wLnNhdmUob3MucGF0aC5qb2luKG91dCwgJ2VjZ18yNTBfaW50MTYubnB5JyksIGJpZykNCiAgICBFID0gcGQuRGF0YUZyYW1lKHJvd3MpDQogICAgRS50b19jc3Yob3MucGF0aC5qb2luKG91dCwgJ2VjZ19yZWNvcmRzLmNzdicpLCBpbmRleD1GYWxzZSkNCiAgICBtYW4gPSBkaWN0KHZlcnNpb249dmVyc2lvbiwgY3JlYXRlZF91dGM9ZGF0ZXRpbWUubm93KHRpbWV6b25lLnV0YykuaXNvZm9ybWF0KHRpbWVzcGVjPSdzZWNvbmRzJyksIGZzPUVDR19GUywNCiAgICAgICAgICAgICAgIGludDE2X3Blcl91bml0PUVDR19JTlQxNl9QRVJfVU5JVCwgYmFuZF9oej1bMC41LCA0MC4wXSwNCiAgICAgICAgICAgICAgIGFsaWdubWVudD0nRUNHIHNhbXBsZSBrIGF0IGsgLyAyNTAgczsgc3BlY3Ryb2dyYW0gZnJhbWUgaiAoY2VudHJlIGogLyAyNTAgKyAwLjAxNiBzKSBwYWlycyB3aXRoIEVDRyBzYW1wbGUgaiArIDQnLA0KICAgICAgICAgICAgICAgcGFja19tYW5pZmVzdF9zaGEyNTY9c2hhMjU2KG9zLnBhdGguam9pbihvdXQsICdtYW5pZmVzdC5qc29uJykpLA0KICAgICAgICAgICAgICAgZmlsZXM9e2Y6IHNoYTI1Nihvcy5wYXRoLmpvaW4ob3V0LCBmKSkgZm9yIGYgaW4gKCdlY2dfMjUwX2ludDE2Lm5weScsICdlY2dfcmVjb3Jkcy5jc3YnKX0sDQogICAgICAgICAgICAgICBuX3JlY29yZHM9bGVuKEUpLCBuX3NhbXBsZXM9aW50KGxlbihiaWcpKSkNCiAgICBqc29uLmR1bXAobWFuLCBvcGVuKG9zLnBhdGguam9pbihvdXQsICdlY2dfbWFuaWZlc3QuanNvbicpLCAndycpLCBpbmRlbnQ9MSkNCiAgICBwcmludCgnRUNHIHRhcmdldHM6ICVkIHJlY29yZHMsICUuMWYgaCwgJS4wZiBNQiwgJS4wZiBzOyByZWNvcmRzIHdpdGggRUNHIHNob3J0ZXIgdGhhbiBQQ0c6ICVkJyAlICgNCiAgICAgICAgbGVuKEUpLCBsZW4oYmlnKSAvIEVDR19GUyAvIDM2MDAsIG9zLnBhdGguZ2V0c2l6ZShvcy5wYXRoLmpvaW4ob3V0LCAnZWNnXzI1MF9pbnQxNi5ucHknKSkgLyAxZTYsIHRpbWUudGltZSgpIC0gdDAsDQogICAgICAgIGludCgoRVsnbl9zaG9ydCddID4gMCkuc3VtKCkpKSkNCiAgICByZXR1cm4gbWFuDQoNCg0KZGVmIGxvYWRfZWNnKGZvbGRlcik6DQogICAgIiIiQ29sYWIgc2lkZTogbWVtb3J5LW1hcHBlZCBFQ0cgdGFyZ2V0cywgY2hlY2tlZCBhZ2FpbnN0IGVjZ19tYW5pZmVzdC5qc29uIGFuZCB0aWVkIHRvIHRoZSBwYWNrJ3MgbWFuaWZlc3QuIiIiDQogICAgbWFuID0ganNvbi5sb2FkKG9wZW4ob3MucGF0aC5qb2luKGZvbGRlciwgJ2VjZ19tYW5pZmVzdC5qc29uJykpKQ0KICAgIGZvciBmLCBoIGluIG1hblsnZmlsZXMnXS5pdGVtcygpOg0KICAgICAgICBhc3NlcnQgc2hhMjU2KG9zLnBhdGguam9pbihmb2xkZXIsIGYpKSA9PSBoLCAnaGFzaCBtaXNtYXRjaDogJXMnICUgZg0KICAgIGFzc2VydCBzaGEyNTYob3MucGF0aC5qb2luKGZvbGRlciwgJ21hbmlmZXN0Lmpzb24nKSkgPT0gbWFuWydwYWNrX21hbmlmZXN0X3NoYTI1NiddLCAnRUNHIGZpbGUgYmVsb25ncyB0byBhbm90aGVyIHBhY2snDQogICAgcmV0dXJuIG1hbiwgbnAubG9hZChvcy5wYXRoLmpvaW4oZm9sZGVyLCAnZWNnXzI1MF9pbnQxNi5ucHknKSwgbW1hcF9tb2RlPSdyJyksIHBkLnJlYWRfY3N2KG9zLnBhdGguam9pbihmb2xkZXIsICdlY2dfcmVjb3Jkcy5jc3YnKSkNCg0KDQppZiBfX25hbWVfXyA9PSAnX19tYWluX18nOg0KICAgIGFwID0gYXJncGFyc2UuQXJndW1lbnRQYXJzZXIoKQ0KICAgIGFwLmFkZF9hcmd1bWVudCgnLS12ZXJzaW9uJywgZGVmYXVsdD1QQUNLKQ0KICAgIGFwLmFkZF9hcmd1bWVudCgnLS1lY2cnLCBhY3Rpb249J3N0b3JlX3RydWUnLCBoZWxwPSdhZGQgdGhlIEVDRyB0YXJnZXRzIGZvciBXIHRvIGFuIGV4aXN0aW5nIHBhY2snKQ0KICAgIGEgPSBhcC5wYXJzZV9hcmdzKCkNCiAgICBidWlsZF9lY2coYS52ZXJzaW9uKSBpZiBhLmVjZyBlbHNlIGJ1aWxkKGEudmVyc2lvbikNCg==", "train.py": "..."}
MOD_SHA = {"__init__.py": "012ad603c7da722a8234aa407cbb3b75ce077f4807d0fd0f96a9b6314b8cde63", "config.py": "44c1796b2abaa55b39de7c4caba121a941b5e789b695e74dacc83d7a6579eb68", "frontend.py": "d40f7c891441982239c8a71577e85abada77a3966619e908b97c331bc6f65760", "augment.py": "663e6ff7409db4dcc9d2156cd1e3a6d3de142b71846f458ef17fd02eb8daf068", "losses.py": "cea4c2edf661889705bf8bdc627d06c545ca631cd1446f0a355d20b302970440", "model.py": "2b73aecb8543b04398cc6ceb2f8a7630662d273b333f3458f6948c6186014b95", "robust_beats.py": "09bb9d7f22105eefd31cd7ae883da95af98e7d05dd8e126c8e20799035d4c077", "decoder.py": "daa8797c9a475e2085b1298ad224563a68080c54ec5c3b7beb47b9b2099913ed", "corpus.py": "fc40a7a9db54292898b0911789b1389aa038eb1f3d3091be7d1ec25068800ba6", "train.py": "06c4afd84894eccf2bc10ac95e914b4dd7dbc504ba6e601325848dbf6d230d82"}
os.makedirs('/content/extstudy', exist_ok=True)
for m, b64 in MOD_B64.items():
    src = base64.b64decode(b64)
    assert hashlib.sha256(src).hexdigest() == MOD_SHA[m], 'module %s corrupted' % m
    open('/content/extstudy/' + m, 'wb').write(src)
if '/content' not in sys.path:
    sys.path.insert(0, '/content')
from extstudy import train as T, corpus as C, model as M
assert 'A' in M.HEADS and M.HEADS['A'] == ('s1', 's2') and T.HEADS_OF['A'] == ('s1', 's2')
print('modules written:', sorted(MOD_SHA))

## Training pack and the acoustic labels: copied to the Colab disk, every hash checked

In [ ]:
def _retry(fn, *a, tries=6):
    for k in range(tries):
        try:
            return fn(*a)
        except OSError as e:
            if k == tries - 1:
                raise
            if e.errno in (errno.ENOTCONN, errno.ECONNABORTED, errno.EIO):
                drive.mount('/content/drive', force_remount=True)
            time.sleep(3 * (k + 1))
SRC = DRIVE_STUDY + '/corpus/paired_v3'
PACK = '/content/pack/paired_v3'
os.makedirs(PACK, exist_ok=True)
for f in ('manifest.json', 'records.csv', 'bin_medians.npy', 'events.npz', 'pcg_bp_1k_int16.npy'):
    if not (os.path.exists(os.path.join(PACK, f)) and os.path.getsize(os.path.join(PACK, f)) == _retry(os.path.getsize, os.path.join(SRC, f))):
        _retry(shutil.copyfile, os.path.join(SRC, f), os.path.join(PACK, f))
assert hashlib.sha256(open(os.path.join(PACK, 'manifest.json'), 'rb').read()).hexdigest() == '21e2671215a15feee3ac96d87a9e4c644a936992ade9da97f2eb3e8516d084fd', 'the pack differs from the one T trained on'
AC_SRC = DRIVE_STUDY + '/corpus/' + AC_VERSION
AC = '/content/pack/' + AC_VERSION
os.makedirs(AC, exist_ok=True)
for f in (EVENTS_FILE, 'laws.json'):
    _retry(shutil.copyfile, os.path.join(AC_SRC, f), os.path.join(AC, f))
assert hashlib.sha256(open(os.path.join(AC, EVENTS_FILE), 'rb').read()).hexdigest() == EVENTS_SHA256, 'acoustic events file differs from the hash'
LAWS = json.load(open(os.path.join(AC, 'laws.json')))
assert LAWS['sha256'] == LAWS_SHA256, 'acoustic laws differ from the hash'
EV = dict(np.load(os.path.join(AC, EVENTS_FILE)))
DATA = T.PairedData(PACK, LAWS, with_ecg=False).replace_events(EV, LAWS)     # the pack's audio and splits; the acoustic labels
n_valid = int(sum(len(DATA.events[k]['vidx']) for k in DATA.events))
print('pack loaded: %d records, %s | acoustic labels: %d beats, %d valid; records in the training index: %d' % (
    len(DATA.R), {r: len(v) for r, v in DATA.roles.items()}, len(EV['r_s']), n_valid,
    sum(len(g) for d in DATA.index['train'].values() for g in d)))

## Train and freeze

In [ ]:
CFG = dict(T.CFG, workers=min(10, os.cpu_count() - 2), eval_workers=min(10, os.cpu_count() - 2))
if SMOKE:
    CFG.update(epochs=1, steps_per_epoch=20, val_max_records_per_dataset=5)
stamp = datetime.datetime.now().strftime('%Y%m%d_%H%M') + '_ac' + ('_smoke' if SMOKE else '')
OUT = os.path.join(DRIVE_STUDY, 'checkpoints', stamp)
os.makedirs(OUT, exist_ok=True)
FREEZE = dict(created=None, smoke=SMOKE, gpu=GPU, cfg={k: v for k, v in CFG.items()}, modules_sha256=MOD_SHA,
              laws_sha256=LAWS['sha256'], label_check=dict(status='REPORTED, NOT REQUIRED (EXP2_PREREG.md)', acoustic_gate=LABEL_GATE),
              acoustic_events=dict(version=AC_VERSION, file=EVENTS_FILE, sha256=EVENTS_SHA256), exp2_plan_sha256=EXP2_PLAN_SHA256,
              pack_manifest_sha256=hashlib.sha256(open(os.path.join(PACK, 'manifest.json'), 'rb').read()).hexdigest(),
              models={})
VAL = {}
for kind in KINDS:
    VAL[kind] = T.validation_set(DATA, kind, CFG)
    for seed in (SEEDS[kind][:1] if SMOKE else SEEDS[kind]):
        print('==== %s seed %d ====' % (kind, seed)); t0 = time.time()
        model, info = T.fit(DATA, kind, seed, DEV, CFG, val_items=VAL[kind])
        state = {k: v.detach().cpu() for k, v in model.state_dict().items()}
        fp = os.path.join(OUT, '%s_seed%d.pt' % (kind, seed))
        torch.save(dict(state_dict=state, info=info, kind=kind, seed=seed), fp)
        FREEZE['models']['%s_seed%d' % (kind, seed)] = dict(weights_sha256=T.state_sha256(state), file_sha256=hashlib.sha256(open(fp, 'rb').read()).hexdigest(),
                                                              best_val=info['best_val'], best_epoch=int(np.argmax([h['val_score'] for h in info['history']]) + 1),
                                                              minutes=(time.time() - t0) / 60)
        print('saved', fp, FREEZE['models']['%s_seed%d' % (kind, seed)])
FREEZE['created'] = datetime.datetime.now(datetime.timezone(datetime.timedelta(hours=5))).isoformat(timespec='seconds')
json.dump(FREEZE, open(os.path.join(OUT, 'freeze.json'), 'w'), indent=1)
print('freeze record written:', os.path.join(OUT, 'freeze.json'))

## Gate A: paired test subjects (descriptive)

Beat-identity F1 of each frozen model on native 30 s crops of the paired test records, per dataset, against a shifted
null (decoded beats moved by half the median RR); kappa = (F1 - null) / (1 - null). Read beside T's values in
`checkpoints/20261018_0908/paired_test.json`.

In [ ]:
def crops(role, kind, seed=777):
    rng = np.random.default_rng(seed); c2 = dict(CFG, crop_frames=CFG['val_crop_frames']); out = []
    for d, groups in sorted(DATA.index[role].items()):
        for g in groups:
            for k in g:
                if DATA.events[k]['runs']:
                    out.append(T.make_sample(DATA, k, rng, c2, augment=False, warp_hr=None, kind=kind))
    return out

def kappa_eval(model, items, kind):
    from extstudy import decoder as D
    model.eval(); res = {}
    with torch.no_grad():
        for it in items:
            o = model(torch.tensor(it['spec'][None], device=DEV)).float().cpu().numpy()[0]
            p = 1 / (1 + np.exp(-o)); dec = D.decode_beats(D.beat_trace_from_heads(p[0], p[1]), 250, kind='event')
            truth = it['s1_times']
            b = it['out_t0'] + dec['beats_s']
            f1 = T.beat_f1(truth, b, it['mask'], it['out_t0'])
            rr = np.median(np.diff(truth)) if len(truth) > 2 else 0.8
            f0 = T.beat_f1(truth, b + rr / 2, it['mask'], it['out_t0'])
            res.setdefault(it['info']['dataset'], []).append((f1, f0))
    out = {}
    for d, v in res.items():
        v = np.array(v, float); f1, f0 = np.nanmean(v[:, 0]), np.nanmean(v[:, 1])
        out[d] = dict(f1=float(f1), null=float(f0), kappa=float((f1 - f0) / (1 - f0)) if f0 < 1 else None, n=len(v))
    return out

GATE_A = {}
for kind in KINDS:
    test_items = crops('test', kind)
    for name, m in FREEZE['models'].items():
        ck = torch.load(os.path.join(OUT, name + '.pt'), map_location=DEV)
        model = M.UNet1D(kind).to(DEV); model.load_state_dict(ck['state_dict'])
        assert T.state_sha256({k: v.cpu() for k, v in model.state_dict().items()}) == m['weights_sha256']
        GATE_A[name] = dict(test=kappa_eval(model, test_items, kind), val_kappa=kappa_eval(model, VAL[kind], kind))
        print(name, json.dumps({d: round(v['kappa'], 3) if v['kappa'] is not None else None for d, v in GATE_A[name]['test'].items()}))
json.dump(GATE_A, open(os.path.join(OUT, 'paired_test.json'), 'w'), indent=1)
print('Gate A written:', os.path.join(OUT, 'paired_test.json'))